In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (use the full available series; do not filter)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# 1) Unique cmdb_id values found in each file with counts per cmdb_id per file
cmdb_counts_metric = metric.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
cmdb_counts_trace = trace.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
cmdb_counts_log = log.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
cmdb_counts_error = error_logs.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# Limit to top 20 rows for display (compact)
cmdb_counts_metric = cmdb_counts_metric.head(20)
cmdb_counts_trace = cmdb_counts_trace.head(20)
cmdb_counts_log = cmdb_counts_log.head(20)
cmdb_counts_error = cmdb_counts_error.head(20)

# 2) For metric.csv: unique (cmdb_id, kpi_name) pairs with number of samples and global stats (P95, median, P5, min, max, count)
# Compute aggregations across the full series (no time filtering)
grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_kpi_stats = grp.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p50=lambda x: x.quantile(0.50),
    p5=lambda x: x.quantile(0.05),
    vmin='min',
    vmax='max'
).reset_index()

# Round numeric columns for compactness
metric_kpi_stats[['p95','p50','p5','vmin','vmax']] = metric_kpi_stats[['p95','p50','p5','vmin','vmax']].round(6)

# Sort by sample count desc and take top 20
metric_kpi_stats_top20 = metric_kpi_stats.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 3) trace.csv: unique (cmdb_id, trace_name) pairs with number of samples (top 20)
trace_pairs = trace.groupby(['cmdb_id', 'trace_name']).size().reset_index(name='count')
trace_pairs_top20 = trace_pairs.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 4) log.csv: unique (cmdb_id, log_name) pairs with number of samples (top 20)
log_pairs = log.groupby(['cmdb_id', 'log_name']).size().reset_index(name='count')
log_pairs_top20 = log_pairs.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# 5) error_logs.csv: counts per cmdb_id and earliest/latest timestamp per cmdb_id (top 20)
err_grp = error_logs.groupby('cmdb_id')['timestamp'].agg(['count', 'min', 'max']).reset_index().rename(columns={'min':'earliest_ts','max':'latest_ts'})
# convert earliest/latest to Asia/Shanghai timezone-aware strings
err_grp['earliest'] = pd.to_datetime(err_grp['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
err_grp['latest'] = pd.to_datetime(err_grp['latest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
error_logs_summary_top20 = err_grp[['cmdb_id','count','earliest','latest']].sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs (limit rows to at most 20)
cmdb_counts_metric, cmdb_counts_trace, cmdb_counts_log, cmdb_counts_error, metric_kpi_stats_top20, trace_pairs_top20, log_pairs_top20, error_logs_summary_top20

```
Out[1]:
```
```python
# Summarize results in plain English using existing loaded dataframes
metric_cmdb_count = metric['cmdb_id'].nunique()
trace_cmdb_count = trace['cmdb_id'].nunique()
log_cmdb_count = log['cmdb_id'].nunique()
err_cmdb_count = error_logs['cmdb_id'].nunique()
metric_top_cmdb = metric.groupby('cmdb_id').size().sort_values(ascending=False).head(5)
trace_pair_count = trace.groupby(['cmdb_id','trace_name']).size().shape[0]
trace_top3 = trace.groupby(['cmdb_id','trace_name']).size().sort_values(ascending=False).head(3)
log_pair_count = log.groupby(['cmdb_id','log_name']).size().shape[0]
log_top3 = log.groupby(['cmdb_id','log_name']).size().sort_values(ascending=False).head(3)
error_log_total = error_logs.shape[0]

# Use the already computed metric_kpi_stats_top20 (top 20 metric (cmdb_id,kpi_name) stats)
top_metric_pairs_sample = metric_kpi_stats_top20[['cmdb_id','kpi_name','count']].head(5)
# pick first row stats for a quick example (if exists)
example_stats = None
if not metric_kpi_stats_top20.empty:
    r = metric_kpi_stats_top20.iloc[0]
    example_stats = {
        'cmdb_id': r['cmdb_id'],
        'kpi_name': r['kpi_name'],
        'count': int(r['count']),
        'p95': float(r['p95']),
        'median': float(r['p50']),
        'p5': float(r['p5']),
        'min': float(r['vmin']),
        'max': float(r['vmax'])
    }

summary = (
    f"Summary of loaded telemetry (plain English):\n\n"
    f"1) Unique cmdb_id counts per file:\n"
    f"   - metric.csv: {metric_cmdb_count} unique cmdb_id(s). Top 5 by sample count: "
    + ", ".join([f"{idx} ({cnt})" for idx,cnt in metric_top_cmdb.items()]) + ".\n"
    f"   - trace.csv: {trace_cmdb_count} unique cmdb_id(s).\n"
    f"   - log.csv: {log_cmdb_count} unique cmdb_id(s).\n"
    f"   - error_logs.csv: {err_cmdb_count} unique cmdb_id(s), total records: {error_log_total}.\n\n"
    f"2) metric.csv (top metric (cmdb_id,kpi_name) pairs by sample count - up to 20):\n"
    f"   - Top sample pairs (showing up to 5):\n"
    f"     {top_metric_pairs_sample.to_dict(orient='records')}\n"
    + (f"   - Example stats for top pair: {example_stats}\n" if example_stats else "")
    f"\n3) trace.csv:\n"
    f"   - Unique (cmdb_id, trace_name) pairs: {trace_pair_count}.\n"
    f"   - Top 3 pairs by sample count: {trace_top3.to_dict()}\n\n"
    f"4) log.csv:\n"
    f"   - Unique (cmdb_id, log_name) pairs: {log_pair_count}.\n"
    f"   - Top 3 pairs by sample count: {log_top3.to_dict()}\n\n"
    f"5) error_logs.csv:\n"
    + ( "   - No error log records were found.\n" if error_log_total==0 else
        f"   - Total records: {error_log_total}. Per-cmdb earliest/latest timestamps are available (top 20 returned earlier).\n")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  count
0         Mysql01   4886
1         Mysql02   4870
2        Tomcat02   2043
3        Tomcat03   2040
4        Tomcat04   2012
5        Tomcat01   1915
6         Redis02   1800
7            MG02   1572
8            MG01   1559
9         Redis01   1509
10           IG01   1100
11           IG02   1085
12       apache02    903
13       apache01    881
14       dockerA2    140
15   ServiceTest1    124
16  ServiceTest10    124
17  ServiceTest11    124
18   ServiceTest4    124
19   ServiceTest8    124,      cmdb_id  count
0       MG02   1176
1       MG01   1176
2   Tomcat03    468
3   Tomcat02    462
4       IG01    459
5       IG02    459
6   Tomcat01    453
7   Tomcat04    453
8   dockerA1    453
9   dockerB1    453
10  dockerB2    453
11  dockerA2    450,     cmdb_id  count
0  Tomcat04     69
1  Tomcat03     68
2  Tomcat02     67
3  Tomcat01     65
4  apache01     62
5  apache02     62
6   Mysql01      2
7   Mysql02      2, Empty DataFrame
Columns: [cmdb_id, count]
Index: [],      cmdb_id                                           kpi_name  count           p95           p50            p5          vmin          vmax
0   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
2   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
4   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00
5   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
6   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
7   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
8   Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
9   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
10  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
12  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
13  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
14  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  7.000000e+02  7.000000e+02  7.000000e+02  7.000000e+02  7.000000e+02
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.136860e+05  4.136860e+05  4.136860e+05  4.136860e+05  4.136860e+05
16  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.325676e+09  1.325568e+09  1.325242e+09  1.325134e+09  1.325685e+09
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.276624e+06  2.275387e+06  2.273704e+06  2.273464e+06  2.276729e+06
18  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
19  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,      cmdb_id                         trace_name  count
0       IG01           trace.self.duration_mean     31
1       IG01            trace.self.duration_p95     31
2   dockerA2      trace.from_MG02.duration_mean     31
3       IG01               trace.self.row_count     31
4   dockerA2           trace.self.duration_mean     31
5   dockerA2          trace.from_MG02.row_count     31
6   dockerA2               trace.self.row_count     31
7   dockerA2       trace.from_MG02.duration_p95     31
8   dockerA2            trace.self.duration_p95     31
9       MG01   trace.from_Tomcat03.duration_p95     31
10      MG01           trace.self.duration_mean     31
11      MG01      trace.from_Tomcat03.row_count     31
12      MG01  trace.from_Tomcat03.duration_mean     31
13      MG01            trace.self.duration_p95     31
14      MG01               trace.self.row_count     31
15  dockerA1               trace.self.row_count     31
16  dockerA1            trace.self.duration_p95     31
17  dockerA1           trace.self.duration_mean     31
18      MG02      trace.from_Tomcat03.row_count     31
19      MG02   trace.from_Tomcat03.duration_p95     31,      cmdb_id          log_name  count
0   apache02   log.error_count     31
1   Tomcat03   log.error_count     31
2   Tomcat02     log.row_count     31
3   Tomcat02   log.error_count     31
4   apache02     log.row_count     31
5   apache01     log.row_count     31
6   Tomcat04   log.error_count     31
7   Tomcat03     log.row_count     31
8   apache01   log.error_count     31
9   Tomcat04     log.row_count     31
10  Tomcat01   log.error_count     30
11  Tomcat01     log.row_count     30
12  Tomcat04  log.gc_pause_max      7
13  Tomcat03  log.gc_pause_max      6
14  Tomcat02  log.gc_pause_max      5
15  Tomcat01  log.gc_pause_max      5
16   Mysql02     log.row_count      1
17   Mysql02   log.error_count      1
18   Mysql01     log.row_count      1
19   Mysql01   log.error_count      1, Empty DataFrame
Columns: [cmdb_id, count, earliest, latest]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse metric variable already loaded in the kernel
# 1) Restrict to allowed root-cause components
allowed_components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                      'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
metric_allowed = metric[metric['cmdb_id'].isin(allowed_components)].copy()

# 2) Compute global statistics across the FULL available series (before any time filtering)
global_stats_df = metric_allowed.groupby(['cmdb_id','kpi_name'])['value'].agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p50=lambda x: x.quantile(0.50),
    p5=lambda x: x.quantile(0.05),
    vmin='min',
    vmax='max'
).reset_index()

# Round numeric stats for compactness (store full precision in memory but present rounded)
for c in ['p95','p50','p5','vmin','vmax']:
    global_stats_df[c] = global_stats_df[c].round(6)

# 3) Filter metric_allowed to the incident time window in UTC+8 (inclusive)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021,3,25,0,0,0))
end_local   = tz.localize(datetime(2021,3,25,0,30,0))
# convert to UTC epoch seconds to compare with timestamp column (which is seconds)
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

metric_window = metric_allowed[(metric_allowed['timestamp'] >= start_ts) & (metric_allowed['timestamp'] <= end_ts)].copy()

# Merge window rows with precomputed global stats (thresholds)
win_with_stats = metric_window.merge(global_stats_df, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_glob'))

# Detect anomalies in-window: upper if value >= p95, lower if value <= p5
win_with_stats['is_upper'] = win_with_stats['value'] >= win_with_stats['p95']
win_with_stats['is_lower'] = win_with_stats['value'] <= win_with_stats['p5']
win_with_stats['is_anomaly'] = win_with_stats['is_upper'] | win_with_stats['is_lower']

# 4) Aggregate anomalies per (cmdb_id,kpi_name)
# First compute per-group counts in window
count_in_window = win_with_stats.groupby(['cmdb_id','kpi_name']).size().reset_index(name='count_in_window')

# Compute anomalies details
anoms = win_with_stats[win_with_stats['is_anomaly']].copy()

if not anoms.empty:
    # earliest anomaly timestamp per group
    grp_earliest = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index(name='earliest_anomaly_timestamp')
    # count anomalies per group
    grp_count_anom = anoms.groupby(['cmdb_id','kpi_name']).size().reset_index(name='count_anomalies')
    # determine if group has any upper anomalies
    grp_has_upper = anoms.groupby(['cmdb_id','kpi_name'])['is_upper'].any().reset_index(name='has_upper')
    # compute max_breach per group depending on type: for upper use max(value-p95); for lower use max(p5-value)
    # compute breaches for each anomaly row
    anoms['upper_breach'] = (anoms['value'] - anoms['p95']).where(anoms['is_upper'], 0.0)
    anoms['lower_breach'] = (anoms['p5'] - anoms['value']).where(anoms['is_lower'], 0.0)
    grp_max_upper = anoms.groupby(['cmdb_id','kpi_name'])['upper_breach'].max().reset_index(name='max_upper_breach')
    grp_max_lower = anoms.groupby(['cmdb_id','kpi_name'])['lower_breach'].max().reset_index(name='max_lower_breach')

    # assemble aggregated anomaly summary
    anomalies_summary = count_in_window.merge(grp_count_anom, on=['cmdb_id','kpi_name'], how='left') \
                                       .merge(grp_earliest, on=['cmdb_id','kpi_name'], how='left') \
                                       .merge(grp_has_upper, on=['cmdb_id','kpi_name'], how='left') \
                                       .merge(grp_max_upper, on=['cmdb_id','kpi_name'], how='left') \
                                       .merge(grp_max_lower, on=['cmdb_id','kpi_name'], how='left')

    # Fill NaNs for groups with zero anomalies (if any)
    anomalies_summary['count_anomalies'] = anomalies_summary['count_anomalies'].fillna(0).astype(int)
    anomalies_summary['has_upper'] = anomalies_summary['has_upper'].fillna(False)

    # determine anomaly_type and max_breach per instructions
    def choose_type_and_breach(row):
        if row['count_anomalies'] == 0:
            return pd.Series([None, 0.0])
        if row['has_upper']:
            return pd.Series(['upper', float(row['max_upper_breach'])])
        else:
            return pd.Series(['lower', float(row['max_lower_breach'])])

    anomalies_summary[['anomaly_type','max_breach']] = anomalies_summary.apply(choose_type_and_breach, axis=1)

    # Keep requested columns and sort by max_breach desc, limit to top 20
    anomalies_summary_top20 = anomalies_summary[['cmdb_id','kpi_name','count_in_window','count_anomalies',
                                                'earliest_anomaly_timestamp','anomaly_type','max_breach']] \
                                    .sort_values('max_breach', ascending=False).head(20).reset_index(drop=True)
else:
    # No anomalies detected in the window
    anomalies_summary_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','count_anomalies',
                                                   'earliest_anomaly_timestamp','anomaly_type','max_breach'])

# 5) Example anomaly records (up to 10) with requested columns
if not anoms.empty:
    anomaly_examples10 = anoms[['cmdb_id','kpi_name','timestamp','value','p95','p5']].sort_values(['timestamp']).head(10).reset_index(drop=True)
else:
    anomaly_examples10 = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p5'])

# Final displayed variables (compact)
global_stats_df.shape, global_stats_df.head(3), anomalies_summary_top20, anomaly_examples10

```
Out[2]:
```
```python
# Summarize findings in plain English using existing kernel variables
n_pairs = int(global_stats_df.shape[0])  # number of (cmdb_id,kpi_name) pairs with global stats
top_rows = anomalies_summary_top20.head(5)
types_count = anomalies_summary_top20['anomaly_type'].value_counts().to_dict() if not anomalies_summary_top20.empty else {}

lines = []
lines.append(f"Summary (plain English):")
lines.append(f"- Computed global statistics (p95, p50, p5, min, max, count) for {n_pairs} (cmdb_id, kpi_name) pairs across the allowed components.")
lines.append(f"- Time window analyzed: 2021-03-25 00:00:00 to 2021-03-25 00:30:00 (UTC+8).")
if anomalies_summary_top20.empty:
    lines.append("- No anomalies detected in the window based on global p95/p5 thresholds.")
else:
    total_returned = anomalies_summary_top20.shape[0]
    lines.append(f"- Detected anomaly groups (top {min(20,total_returned)} returned). Anomaly type counts among returned groups: {types_count}.")
    lines.append("- Top 5 anomaly groups (sorted by max_breach):")
    for i, r in top_rows.iterrows():
        lines.append(
            f"  {i+1}. {r['cmdb_id']} | {r['kpi_name'][:80]}{'...' if len(r['kpi_name'])>80 else ''} "
            f"| in_window={int(r['count_in_window'])} anomalies={int(r['count_anomalies'])} "
            f"| earliest_anom_ts={int(r['earliest_anomaly_timestamp'])} "
            f"| type={r['anomaly_type']} | max_breach={float(r['max_breach']):.1f}"
        )
    lines.append("- Additionally, up to 10 example anomaly records were prepared for inspection (showing cmdb_id, kpi_name, timestamp, value, p95, p5).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((1707, 8),   cmdb_id                                    kpi_name  count           p95           p50            p5          vmin          vmax
0    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
1    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsage     19  3.937683e+01  1.821860e+01  5.341580e+00  3.990500e+00  3.953010e+01
2    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.405232e+08  4.360238e+08  1.290679e+08  8.141075e+07  8.849382e+08,      cmdb_id                                           kpi_name  count_in_window  count_anomalies  earliest_anomaly_timestamp anomaly_type  max_breach
0   Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                6                2                  1616602860        upper  86343124.0
1       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31                4                  1616601720        upper  85292672.0
2   Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                6                2                  1616602860        upper  68343234.0
3    Mysql02               Mysql-MySQL_3306_Innodb data written               31                4                  1616602800        upper  47545344.0
4       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed               31                4                  1616602020        upper  44414940.0
5       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed               31                4                  1616602140        upper  32484872.0
6   Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                6                2                  1616601660        upper  26292394.0
7   Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                6                2                  1616603040        upper  25795710.0
8   Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                4                2                  1616601600        upper  24955556.4
9    Mysql02             Mysql-MySQL_3306_Innodb os log written               31                4                  1616601660        upper  24851968.0
10  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                3                2                  1616602380        upper  20343342.4
11  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                3                2                  1616601720        upper  16529967.2
12   Mysql01               Mysql-MySQL_3306_Innodb data written               31                4                  1616603100        upper  14866688.0
13      MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31                4                  1616601840        upper  13679264.0
14  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                6                2                  1616601960        upper   9982844.0
15   Mysql02                        Mysql-MySQL_3306_Bytes Sent               31                4                  1616603100        upper   5599514.5
16   Mysql01                        Mysql-MySQL_3306_Bytes Sent               31                4                  1616601660        upper   1470005.0
17   Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests               31                4                  1616603100        upper    275138.5
18   Mysql01             Mysql-MySQL_3306_Innodb os log written               31                4                  1616601660        upper    221184.0
19   Mysql02  Mysql-MySQL_3306_Innodb buffer pool write requ...               31                4                  1616601660        upper    111886.5,     cmdb_id                                           kpi_name   timestamp         value           p95            p5
0  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...  1616601600  0.000000e+00  0.000000e+00  0.000000e+00
1  Tomcat03  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1616601600  3.000000e-01  3.000000e-01  3.000000e-01
2  Tomcat03         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1616601600  3.000000e+00  3.000000e+00  3.000000e+00
3  Tomcat03    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent  1616601600  0.000000e+00  0.000000e+00  0.000000e+00
4  Tomcat03                       OSLinux-OSLinux_SWAP_SWAP_Si  1616601600  0.000000e+00  0.000000e+00  0.000000e+00
5  Tomcat03                       OSLinux-OSLinux_SWAP_SWAP_So  1616601600  0.000000e+00  0.000000e+00  0.000000e+00
6  Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime  1616601600  7.800542e+06  7.802202e+06  7.800629e+06
7  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1616601600  1.093000e+03  1.093000e+03  1.093000e+03
8  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  1616601600  4.016560e+05  4.016560e+05  4.016560e+05
9  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1616601600  2.043573e+09  2.044141e+09  2.043672e+09)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing variables: metric, global_stats_df

# 1) Filter metric.csv to the incident window March 25, 2021 00:00:00 — 00:30:00 (UTC+8).
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021,3,25,0,0,0))
end_local   = tz.localize(datetime(2021,3,25,0,30,0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 2) Join p95 and p5 from global_stats_df and mark anomalies
# Ensure global_stats_df has the threshold columns
stats_thr = global_stats_df[['cmdb_id','kpi_name','p95','p5']].copy()

win = metric_window.merge(stats_thr, on=['cmdb_id','kpi_name'], how='left')

# Drop rows without thresholds (can't evaluate)
win = win.dropna(subset=['p95','p5']).copy()

# Mark anomalies: upper if value >= p95, lower if value <= p5. Upper takes precedence if both.
win['is_upper'] = win['value'] >= win['p95']
win['is_lower'] = win['value'] <= win['p5']
win['type'] = pd.NA
win.loc[win['is_upper'], 'type'] = 'upper'
win.loc[win['is_lower'] & win['type'].isna(), 'type'] = 'lower'

# Keep only anomaly rows
anoms = win[win['type'].notna()].copy()

# 3) Split anomaly rows into fault segments per (cmdb_id,kpi_name) where consecutive timestamps differ by <= 60s.
if not anoms.empty:
    # sort and compute gaps per group
    anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
    # assign segment ids per group
    def assign_segments(df):
        df = df.sort_values('timestamp').copy()
        df['prev_ts'] = df['timestamp'].shift(1)
        df['gap'] = df['timestamp'] - df['prev_ts']
        # start new segment when gap > 60 or first row
        df['new_seg'] = (df['gap'] > 60) | (df['gap'].isna())
        df['seg_idx'] = df['new_seg'].cumsum()  # relative within full series; will combine with group keys
        return df

    anoms = anoms.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_segments).reset_index(drop=True)

    # Build segment-level aggregations
    seg_grp = anoms.groupby(['cmdb_id','kpi_name','seg_idx'], as_index=False)

    segs = seg_grp.agg(
        segment_start_ts=('timestamp','min'),
        segment_end_ts=('timestamp','max'),
        segment_length_points=('timestamp','size'),
        count_anomalies=('timestamp','size'),
        max_value_in_segment=('value','max'),
        has_upper=('is_upper','any'),
        p95=('p95','first'),
        p5=('p5','first')
    )

    # Determine anomaly_type, max_breach, breach_ratio
    def compute_breach(row):
        if row['has_upper']:
            max_breach = anoms.loc[
                (anoms['cmdb_id']==row['cmdb_id']) &
                (anoms['kpi_name']==row['kpi_name']) &
                (anoms['seg_idx']==row['seg_idx']),
                'value'
            ].sub(row['p95']).max()
            denom = max(abs(row['p95']), 1.0)
            return pd.Series(['upper', float(max_breach), float(max_breach)/denom])
        else:
            max_breach = row['p5'] - anoms.loc[
                (anoms['cmdb_id']==row['cmdb_id']) &
                (anoms['kpi_name']==row['kpi_name']) &
                (anoms['seg_idx']==row['seg_idx']),
                'value'
            ].min()
            denom = max(abs(row['p5']), 1.0)
            return pd.Series(['lower', float(max_breach), float(max_breach)/denom])

    segs[['anomaly_type','max_breach','breach_ratio']] = segs.apply(compute_breach, axis=1)

    # Keep requested columns and sort by max_breach desc, limit to top 20
    segments_top20 = segs[['cmdb_id','kpi_name','segment_start_ts','segment_end_ts','segment_length_points',
                          'anomaly_type','count_anomalies','max_value_in_segment','p95','p5','max_breach','breach_ratio']] \
                      .sort_values('max_breach', ascending=False).head(20).reset_index(drop=True)

    # 4) Up to 10 example anomaly rows for verification
    anomaly_examples10 = anoms[['cmdb_id','kpi_name','timestamp','value','p95','p5','type']].head(10).reset_index(drop=True)

else:
    # No anomalies in window
    segments_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','segment_start_ts','segment_end_ts','segment_length_points',
                                          'anomaly_type','count_anomalies','max_value_in_segment','p95','p5','max_breach','breach_ratio'])
    anomaly_examples10 = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p5','type'])

# Final compact outputs (saved in kernel variables)
segments_top20, anomaly_examples10

```
Out[3]:
```
Summary (plain English):

- I detected fault segments (contiguous per-minute anomaly runs) within the incident window 2021-03-25 00:00–00:30 (UTC+8). I return the top 20 segments sorted by max_breach — listed below are the most important findings.

Top segments (sorted by max_breach, top 5 shown):
1. Tomcat03 | Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory
   - segment: 1616603160 — 1616603160 (1 point)
   - type: upper
   - count_anomalies: 1
   - max_value_in_segment: 1.959124e+09
   - p95: 1.872781e+09
   - max_breach: 86,343,124.0 (breach_ratio 0.0461)

2. Tomcat03 | Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory
   - segment: 1616603340 — 1616603340 (1 point)
   - type: lower
   - count_anomalies: 1
   - max_value_in_segment: 2.228495e+09
   - p5: 2.314838e+09
   - max_breach: 86,343,124.0 (breach_ratio 0.0373)

3. MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed
   - segment: 1616602260 — 1616602260 (1 point)
   - type: upper
   - count_anomalies: 1
   - max_value_in_segment: 1.030803e+09
   - p95: 9.455101e+08
   - max_breach: 85,292,672.0 (breach_ratio 0.0902)

4. Tomcat02 | Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory
   - segment: 1616603160 — 1616603160 (1 point)
   - type: lower
   - count_anomalies: 1
   - max_value_in_segment: 1.636306e+09
   - p5: 1.710680e+09
   - max_breach: 74,373,402.0 (breach_ratio 0.0435)

5. Tomcat01 | Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory
   - segment: 1616603160 — 1616603160 (1 point)
   - type: upper
   - count_anomalies: 1
   - max_value_in_segment: 2.201214e+09
   - p95: 2.132871e+09
   - max_breach: 68,343,234.0 (breach_ratio 0.0320)

Additional observations:
- Many top segments are JVM/memory related (JVMUsedMemory, JVMFreeMemory, HeapMemoryUsed/Max), indicating memory-related anomalies across Tomcat, MG, IG and MySQL components.
- Most fault segments in the returned top 20 are single-minute points; a few are 2-point segments (contiguous anomalies) — e.g., IG01 and IG02 HeapMemoryUsed segments with length 2.
- Example anomaly rows (up to 10) were produced for verification; they include repeated IG01 JVM heap max anomalies across multiple consecutive timestamps (several timestamps where value reached the global threshold).

If you want, I can:
- Expand the example anomaly rows (show p95/p5 and timestamps in human-readable UTC+8),
- Or produce the full top-20 segments table as CSV/JSON for download/inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                   kpi_name  segment_start_ts  segment_end_ts  segment_length_points anomaly_type  count_anomalies  max_value_in_segment           p95            p5  max_breach  breach_ratio
0   Tomcat03    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616603160      1616603160                      1        upper                1          1.959124e+09  1.872781e+09  1.423657e+09  86343124.0      0.046104
1   Tomcat03    Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory        1616603340      1616603340                      1        lower                1          2.228495e+09  2.763963e+09  2.314838e+09  86343124.0      0.037300
2       MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed        1616602260      1616602260                      1        upper                1          1.030803e+09  9.455101e+08  2.248258e+08  85292672.0      0.090208
3   Tomcat02    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616603160      1616603160                      1        lower                1          1.636306e+09  2.208648e+09  1.710680e+09  74373402.0      0.043476
4   Tomcat01    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616603160      1616603160                      1        upper                1          2.201214e+09  2.132871e+09  1.707322e+09  68343234.0      0.032043
5   Tomcat04    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616601660      1616601660                      1        lower                1          1.363196e+09  1.928583e+09  1.422153e+09  58956602.0      0.041456
6    Mysql02       Mysql-MySQL_3306_Innodb data written        1616603280      1616603280                      1        upper                1          2.811098e+12  2.811050e+12  8.000427e+04  47545344.0      0.000017
7       IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed        1616602020      1616602080                      2        upper                2          8.849382e+08  8.405232e+08  1.290679e+08  44414940.0      0.052842
8   Tomcat04    Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory        1616601600      1616601600                      1        lower                1          2.068226e+09  2.563641e+09  2.108679e+09  40453105.2      0.019184
9       MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed        1616602260      1616602260                      1        lower                1          1.729345e+08  9.824805e+08  2.120615e+08  39127044.0      0.184508
10  Tomcat01    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616602860      1616602860                      1        lower                1          1.672361e+09  2.132871e+09  1.707322e+09  34960976.0      0.020477
11      IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed        1616602740      1616602800                      2        upper                2          8.745785e+08  8.405232e+08  1.290679e+08  34055276.0      0.040517
12      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed        1616602140      1616602200                      2        upper                2          8.696219e+08  8.371370e+08  1.034581e+08  32484872.0      0.038805
13      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed        1616602860      1616602920                      2        upper                2          8.639921e+08  8.371370e+08  1.034581e+08  26855104.0      0.032080
14  Tomcat04    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616602860      1616602860                      1        upper                1          1.954875e+09  1.928583e+09  1.422153e+09  26292394.0      0.013633
15      MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed        1616602920      1616602920                      1        upper                1          9.717615e+08  9.455101e+08  2.248258e+08  26251432.0      0.027764
16  Tomcat03    Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory        1616602860      1616602860                      1        lower                1          1.397861e+09  1.872781e+09  1.423657e+09  25795710.0      0.018119
17  Tomcat03    Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory        1616603040      1616603040                      1        upper                1          2.789758e+09  2.763963e+09  2.314838e+09  25795710.0      0.009333
18  Tomcat04    Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory        1616602200      1616602200                      1        upper                1          2.588597e+09  2.563641e+09  2.108679e+09  24955556.4      0.009734
19   Mysql02     Mysql-MySQL_3306_Innodb os log written        1616603280      1616603280                      1        upper                1          7.407253e+11  7.407005e+11  7.703467e+04  24851968.0      0.000034,   cmdb_id                                  kpi_name   timestamp         value           p95            p5   type
0    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601600  2.040136e+09  2.040136e+09  2.040136e+09  upper
1    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601660  2.040136e+09  2.040136e+09  2.040136e+09  upper
2    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601720  2.040136e+09  2.040136e+09  2.040136e+09  upper
3    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601780  2.040136e+09  2.040136e+09  2.040136e+09  upper
4    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601840  2.040136e+09  2.040136e+09  2.040136e+09  upper
5    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601900  2.040136e+09  2.040136e+09  2.040136e+09  upper
6    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616601960  2.040136e+09  2.040136e+09  2.040136e+09  upper
7    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616602020  2.040136e+09  2.040136e+09  2.040136e+09  upper
8    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616602080  2.040136e+09  2.040136e+09  2.040136e+09  upper
9    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryMax  1616602140  2.040136e+09  2.040136e+09  2.040136e+09  upper)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing kernel variables where possible (anoms, metric_window, global_stats_df).
# If 'anoms' is not present, recompute it by joining metric_window with global_stats_df and selecting anomalies.

try:
    anoms  # use existing
except NameError:
    # Recompute anomaly rows for the incident window
    # Assumes metric_window and global_stats_df exist in kernel (from previous steps)
    stats_thr = global_stats_df[['cmdb_id','kpi_name','p95','p5']].copy()
    win = metric_window.merge(stats_thr, on=['cmdb_id','kpi_name'], how='left')
    win = win.dropna(subset=['p95','p5']).copy()
    win['is_upper'] = win['value'] >= win['p95']
    win['is_lower'] = win['value'] <= win['p5']
    win['type'] = pd.NA
    win.loc[win['is_upper'], 'type'] = 'upper'
    win.loc[win['is_lower'] & win['type'].isna(), 'type'] = 'lower'
    anoms = win[win['type'].notna()].copy()

# If no anomalies, prepare empty compact outputs
if anoms.empty:
    timestamp_agg_top10 = pd.DataFrame(columns=['timestamp','total_anomaly_count','number_of_unique_components',
                                                'list_of_unique_components','top_5_components_by_anomaly_count'])
    timestamps_with_2plus = pd.DataFrame(columns=['timestamp','total_anomaly_count'])
    anomaly_examples10 = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p5','type'])
else:
    # 1) Aggregate anomalies by timestamp (minute)
    # First compute per-timestamp per-component counts
    comp_counts = anoms.groupby(['timestamp','cmdb_id']).size().reset_index(name='comp_count')

    # Summarize per timestamp
    def summarize_ts(df):
        total = int(df['comp_count'].sum())
        unique = int(df['cmdb_id'].nunique())
        # list unique components limited to top 10 by comp_count desc
        comp_order = df.sort_values('comp_count', ascending=False)['cmdb_id'].astype(str).unique()
        comps_list = ','.join(comp_order[:10])
        # top 5 components by anomaly count formatted as "component:count"
        top5 = df.sort_values('comp_count', ascending=False).head(5)
        top5s = ';'.join([f"{r['cmdb_id']}:{int(r['comp_count'])}" for _, r in top5.iterrows()])
        return pd.Series({
            'total_anomaly_count': total,
            'number_of_unique_components': unique,
            'list_of_unique_components': comps_list,
            'top_5_components_by_anomaly_count': top5s
        })

    timestamp_agg = comp_counts.groupby('timestamp', group_keys=False).apply(summarize_ts).reset_index()

    # Limit to top 10 timestamps sorted by total_anomaly_count desc
    timestamp_agg_top10 = timestamp_agg.sort_values('total_anomaly_count', ascending=False).head(10).reset_index(drop=True)

    # 2) Chronological list (ascending) of timestamps with total_anomaly_count >= 2 (limit 20)
    timestamps_with_2plus = timestamp_agg[timestamp_agg['total_anomaly_count'] >= 2] \
                                .sort_values('timestamp', ascending=True) \
                                .head(20) \
                                .reset_index(drop=True)[['timestamp','total_anomaly_count']]

    # 3) Up to 10 example anomaly rows for inspection
    anomaly_examples10 = anoms[['cmdb_id','kpi_name','timestamp','value','p95','p5','type']] \
                            .sort_values('timestamp', ascending=True) \
                            .head(10) \
                            .reset_index(drop=True)

# Final compact outputs (DataFrames)
timestamp_agg_top10, timestamps_with_2plus, anomaly_examples10

```
Out[4]:
```
```python
# Prepare a concise plain-English summary using previously computed compact tables:
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

def ts_to_local(ts):
    return datetime.fromtimestamp(int(ts), pytz.utc).astimezone(tz).strftime('%Y-03-%d %H:%M:%S %Z')

# Use existing compact tables in kernel: timestamp_agg_top10, timestamps_with_2plus, anomaly_examples10
top_ts = timestamp_agg_top10.head(5)
two_plus = timestamps_with_2plus
examples = anomaly_examples10.head(5)

lines = []
lines.append("Anomaly timeline summary (incident window: 2021-03-25 00:00–00:30 UTC+8):")
lines.append("")
lines.append("1) Top timestamps by total anomaly count (top 5):")
for _, r in top_ts.iterrows():
    human = ts_to_local(r['timestamp'])
    comps_preview = r['list_of_unique_components']
    top5 = r['top_5_components_by_anomaly_count']
    lines.append(f"   - {human} (epoch {int(r['timestamp'])}): total_anomalies={int(r['total_anomaly_count'])}, unique_components={int(r['number_of_unique_components'])}")
    lines.append(f"       components (sample, up to 10): {comps_preview}")
    lines.append(f"       top contributors: {top5}")
lines.append("")

lines.append("2) Timestamps with >=2 anomalies (chronological, sample up to 20):")
if two_plus.empty:
    lines.append("   - None")
else:
    # show up to 20 timestamps with counts
    for _, r in two_plus.iterrows():
        lines.append(f"   - {ts_to_local(r['timestamp'])} (epoch {int(r['timestamp'])}): total_anomalies={int(r['total_anomaly_count'])}")
lines.append("")

lines.append("3) Example anomaly rows (up to 5):")
if examples.empty:
    lines.append("   - None")
else:
    for _, ex in examples.iterrows():
        lines.append(
            f"   - {ex['cmdb_id']} | {ex['kpi_name'][:80]}{'...' if len(ex['kpi_name'])>80 else ''} | "
            f"{ts_to_local(ex['timestamp'])} (epoch {int(ex['timestamp'])}) | value={ex['value']:.6g} | p95={ex['p95']:.6g} | p5={ex['p5']:.6g} | type={ex['type']}"
        )

lines.append("")
lines.append("Key takeaways:")
lines.append(" - Very large, system-wide anomaly bursts occurred repeatedly during the window: top minutes had ~700–900 anomaly events each.")
lines.append(" - 14 components participated at peak minutes; MySQL nodes (Mysql01, Mysql02) are major contributors in many top minutes.")
lines.append(" - Anomalies span CPU, JVM/memory, Tomcat requests/sessions, and MySQL metrics — both upper and lower breaches are present.")
lines.append(" - The chronological list of minutes with >=2 anomalies can be used to locate clusters; these minutes are concentrated across the window (see list above).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp  total_anomaly_count  number_of_unique_components                          list_of_unique_components                  top_5_components_by_anomaly_count
0  1616602200                  888                           14  Mysql01,Mysql02,Tomcat03,Tomcat04,Tomcat01,Tom...  Mysql01:144;Mysql02:88;Tomcat03:73;Tomcat04:73...
1  1616601720                  809                           14  Mysql01,Mysql02,Redis02,Tomcat03,Tomcat02,Tomc...  Mysql01:127;Mysql02:85;Redis02:81;Tomcat03:73;...
2  1616601600                  802                           14  Mysql01,Mysql02,Tomcat04,Tomcat03,Tomcat02,MG0...  Mysql01:117;Mysql02:100;Tomcat04:74;Tomcat03:7...
3  1616601840                  797                           14  Mysql01,Mysql02,Tomcat02,Tomcat04,Tomcat03,Tom...  Mysql01:142;Mysql02:94;Tomcat02:71;Tomcat04:62...
4  1616602560                  790                           14  Mysql01,Mysql02,Tomcat03,Tomcat01,Redis02,Redi...  Mysql01:126;Mysql02:87;Tomcat03:71;Tomcat01:71...
5  1616602080                  758                           14  Mysql01,Mysql02,Tomcat02,Tomcat04,Redis02,MG01...  Mysql01:115;Mysql02:90;Tomcat02:71;Tomcat04:70...
6  1616603340                  756                           14  Mysql01,Mysql02,Tomcat04,Tomcat02,MG01,MG02,IG...  Mysql01:139;Mysql02:103;Tomcat04:93;Tomcat02:8...
7  1616601960                  707                           14  Mysql01,Mysql02,Redis02,Tomcat01,Tomcat03,Tomc...  Mysql01:115;Mysql02:96;Redis02:72;Tomcat01:62;...
8  1616603280                  706                           14  Mysql01,Mysql02,Tomcat03,Tomcat01,Redis01,Redi...  Mysql01:139;Mysql02:126;Tomcat03:67;Tomcat01:6...
9  1616603220                  705                           14  Mysql02,Mysql01,Tomcat02,Tomcat04,MG01,MG02,To...  Mysql02:150;Mysql01:139;Tomcat02:66;Tomcat04:6...,      timestamp  total_anomaly_count
0   1616601600                  802
1   1616601660                  526
2   1616601720                  809
3   1616601780                  500
4   1616601840                  797
5   1616601900                  403
6   1616601960                  707
7   1616602020                  432
8   1616602080                  758
9   1616602140                  451
10  1616602200                  888
11  1616602260                  400
12  1616602320                  596
13  1616602380                  489
14  1616602440                  676
15  1616602500                  439
16  1616602560                  790
17  1616602620                  456
18  1616602680                  572
19  1616602740                  591,     cmdb_id                                           kpi_name   timestamp         value           p95            p5   type
0  Tomcat01                    OSLinux-CPU_CPU-2_SingleCpuidle  1616601600  8.591385e+01  9.916057e+01  9.491960e+01  lower
1  Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil  1616601600  9.257400e+00  6.387795e+00  9.107350e-01  upper
2  Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuidle  1616601600  9.074259e+01  9.905222e+01  9.552531e+01  lower
3  Tomcat01                        OSLinux-CPU_CPU_CPUUserTime  1616601600  2.542197e+01  2.541870e+01  2.516693e+01  upper
4   Mysql02                        Mysql-MySQL_3306_Sort Range  1616601600  0.000000e+00  4.391400e+04  0.000000e+00  lower
5      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1616601600  3.000000e+00  3.000000e+00  3.000000e+00  upper
6  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1616601600  1.325134e+09  1.325676e+09  1.325242e+09  lower
7  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1616601600  2.273464e+06  2.276624e+06  2.273704e+06  lower
8  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  1616601600  0.000000e+00  0.000000e+00  0.000000e+00  upper
9  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions  1616601600  0.000000e+00  0.000000e+00  0.000000e+00  upper)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse loaded dataframes: trace, log, error_logs (from kernel state)
# Define cluster timestamp windows (given)
cluster1_ts = [1616601660, 1616601720, 1616601780]  # 1616601720 ±60s
cluster2_ts = [1616602140, 1616602200, 1616602260]  # 1616602200 ±60s
cluster_ts_set = set(cluster1_ts + cluster2_ts)

# 1) Trace preprocessing: global thresholds across full trace.csv (do NOT filter by window)
trace_global_stats = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p50=lambda x: x.quantile(0.50),
    p5=lambda x: x.quantile(0.05),
    vmin='min',
    vmax='max'
).reset_index()

# Round for compactness but keep full values in memory
trace_global_stats[['p95','p50','p5','vmin','vmax']] = trace_global_stats[['p95','p50','p5','vmin','vmax']].round(6)

# 2) Trace anomaly detection: filter to cluster timestamps and join thresholds
trace_window = trace[trace['timestamp'].isin(cluster_ts_set)].copy()
trace_win_with_stats = trace_window.merge(
    trace_global_stats[['cmdb_id','trace_name','p95','p5']],
    on=['cmdb_id','trace_name'],
    how='left'
)
# drop rows without thresholds
trace_win_with_stats = trace_win_with_stats.dropna(subset=['p95','p5']).copy()
# mark anomalies: upper if value >= p95 else lower if value <= p5
trace_win_with_stats['is_upper'] = trace_win_with_stats['value'] >= trace_win_with_stats['p95']
trace_win_with_stats['is_lower'] = trace_win_with_stats['value'] <= trace_win_with_stats['p5']
trace_win_with_stats['type'] = pd.NA
trace_win_with_stats.loc[trace_win_with_stats['is_upper'], 'type'] = 'upper'
trace_win_with_stats.loc[trace_win_with_stats['is_lower'] & trace_win_with_stats['type'].isna(), 'type'] = 'lower'
trace_anoms = trace_win_with_stats[trace_win_with_stats['type'].notna()].copy()

# 3) Compact trace-anomaly table (limit 50 rows) and aggregated summary per cluster timestamp
trace_anom_table = trace_anoms[['cmdb_id','trace_name','timestamp','value','p95','p5','type']].sort_values(['timestamp','cmdb_id','trace_name']).head(50).reset_index(drop=True)

# Aggregation per timestamp
if not trace_anoms.empty:
    # per-timestamp & component counts
    trace_comp_counts = trace_anoms.groupby(['timestamp','cmdb_id']).size().reset_index(name='comp_count')
    def summarize_trace_ts(df):
        total = int(df['comp_count'].sum())
        unique_count = int(df['cmdb_id'].nunique())
        comp_order = df.sort_values('comp_count', ascending=False)['cmdb_id'].astype(str).unique()
        comps_list = ','.join(comp_order[:10])
        # top 5 trace_names by anomaly count at this timestamp
        top5 = (trace_anoms[trace_anoms['timestamp']==df['timestamp'].iloc[0]]
                .groupby('trace_name').size().reset_index(name='cnt')
                .sort_values('cnt', ascending=False).head(5))
        top5s = ';'.join([f"{r['trace_name']}:{int(r['cnt'])}" for _,r in top5.iterrows()])
        return pd.Series({
            'total_trace_anomalies': total,
            'unique_trace_components_count': unique_count,
            'list_of_unique_components': comps_list,
            'top_5_trace_names_by_anomaly_count': top5s
        })
    # build per-timestamp summary by using trace_comp_counts grouped by timestamp
    trace_agg = trace_comp_counts.groupby('timestamp').apply(summarize_trace_ts).reset_index()
    # Keep only our cluster timestamps, sort by total anomalies desc for compact view
    trace_agg_summary = trace_agg.sort_values('total_trace_anomalies', ascending=False).reset_index(drop=True)
else:
    trace_agg_summary = pd.DataFrame(columns=['timestamp','total_trace_anomalies','unique_trace_components_count','list_of_unique_components','top_5_trace_names_by_anomaly_count'])

# 4) Log preprocessing: global thresholds across full log.csv
log_global_stats = log.groupby(['cmdb_id','log_name'])['value'].agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    p50=lambda x: x.quantile(0.50),
    p5=lambda x: x.quantile(0.05),
    vmin='min',
    vmax='max'
).reset_index()
log_global_stats[['p95','p50','p5','vmin','vmax']] = log_global_stats[['p95','p50','p5','vmin','vmax']].round(6)

# 5) Log anomaly detection: filter to same cluster timestamps and join thresholds
log_window = log[log['timestamp'].isin(cluster_ts_set)].copy()
log_win_with_stats = log_window.merge(
    log_global_stats[['cmdb_id','log_name','p95','p5']],
    on=['cmdb_id','log_name'],
    how='left'
)
log_win_with_stats = log_win_with_stats.dropna(subset=['p95','p5']).copy()
log_win_with_stats['is_upper'] = log_win_with_stats['value'] >= log_win_with_stats['p95']
log_win_with_stats['is_lower'] = log_win_with_stats['value'] <= log_win_with_stats['p5']
log_win_with_stats['type'] = pd.NA
log_win_with_stats.loc[log_win_with_stats['is_upper'], 'type'] = 'upper'
log_win_with_stats.loc[log_win_with_stats['is_lower'] & log_win_with_stats['type'].isna(), 'type'] = 'lower'
log_anoms = log_win_with_stats[log_win_with_stats['type'].notna()].copy()

# 6) Compact log-anomaly table (limit 50 rows) and aggregated per-cluster summaries
log_anom_table = log_anoms[['cmdb_id','log_name','timestamp','value','p95','p5','type']].sort_values(['timestamp','cmdb_id','log_name']).head(50).reset_index(drop=True)

if not log_anoms.empty:
    # per-timestamp & component counts for logs
    log_comp_counts = log_anoms.groupby(['timestamp','cmdb_id']).size().reset_index(name='comp_count')
    def summarize_log_ts(df):
        total = int(df['comp_count'].sum())
        unique_count = int(df['cmdb_id'].nunique())
        comp_order = df.sort_values('comp_count', ascending=False)['cmdb_id'].astype(str).unique()
        comps_list = ','.join(comp_order[:10])
        return pd.Series({
            'total_log_anomalies': total,
            'unique_log_components_count': unique_count,
            'list_of_unique_components': comps_list
        })
    log_agg = log_comp_counts.groupby('timestamp').apply(summarize_log_ts).reset_index()
    # identify components with log.error_count > 0 at these timestamps (use original log_window rows)
    error_count_positive = log_window[(log_window['log_name']=='log.error_count') & (log_window['value']>0)]
    # aggregated list per timestamp
    errpos_agg = error_count_positive.groupby('timestamp').agg(
        components_with_error_count = ('cmdb_id', lambda x: ','.join(sorted(x.astype(str).unique()))),
        count_components_with_error = ('cmdb_id', lambda x: x.nunique())
    ).reset_index()
    # combine summaries
    log_agg_summary = log_agg.merge(errpos_agg, on='timestamp', how='left').fillna({'components_with_error_count':'','count_components_with_error':0})
else:
    log_agg_summary = pd.DataFrame(columns=['timestamp','total_log_anomalies','unique_log_components_count','list_of_unique_components','components_with_error_count','count_components_with_error'])

# 7) Error log check: search error_logs for records in the cluster windows and return up to 20 rows
error_logs_hits = error_logs[error_logs['timestamp'].isin(cluster_ts_set)].copy()[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# Final compact outputs (saved in kernel variables for inspection)
# Display shapes and small previews for global stats, and the compact anomaly tables and summaries
trace_global_stats.shape, trace_global_stats.head(3), trace_anom_table, trace_agg_summary, log_global_stats.shape, log_global_stats.head(3), log_anom_table, log_agg_summary, error_logs_hits

```
Out[5]:
```
```python
# Summarize trace & log analysis for the two candidate clusters in plain English.
# Uses previously computed kernel tables: trace_global_stats, trace_anom_table, trace_agg_summary,
# log_global_stats, log_anom_table, log_agg_summary, error_logs_hits.

trace_pairs_count = int(trace_global_stats.shape[0]) if 'trace_global_stats' in globals() else 0
log_pairs_count = int(log_global_stats.shape[0]) if 'log_global_stats' in globals() else 0

# Extract cluster summaries (safely)
def get_trace_cluster_summary(ts):
    df = trace_agg_summary
    row = df[df['timestamp']==ts]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        'timestamp': int(ts),
        'total_trace_anomalies': int(r['total_trace_anomalies']),
        'unique_trace_components_count': int(r['unique_trace_components_count']),
        'sample_components_list': str(r['list_of_unique_components']),
        'top_trace_names': str(r['top_5_trace_names_by_anomaly_count'])
    }

def get_log_cluster_summary(ts):
    df = log_agg_summary
    row = df[df['timestamp']==ts]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        'timestamp': int(ts),
        'total_log_anomalies': int(r['total_log_anomalies']),
        'unique_log_components_count': int(r['unique_log_components_count']),
        'sample_components_list': str(r['list_of_unique_components']),
        'components_with_error_count': str(r.get('components_with_error_count','')),
        'count_components_with_error': int(r.get('count_components_with_error',0))
    }

c1 = get_trace_cluster_summary(1616601720)
c2 = get_trace_cluster_summary(1616602200)
l1 = get_log_cluster_summary(1616601720)
l2 = get_log_cluster_summary(1616602200)

# Note whether any matching error_logs found
error_hits_count = 0
if 'error_logs_hits' in globals():
    error_hits_count = int(error_logs_hits.shape[0])

summary = (
    "Diagnosis summary for candidate clusters (plain English):\n\n"
    "Trace preprocessing:\n"
    f"- Computed global trace thresholds for {trace_pairs_count} (cmdb_id, trace_name) pairs (p95,p50,p5,min,max,count).\n\n"
    "Trace anomalies (cluster windows ±60s):\n"
    f"- Cluster around 1616601720: "
    + (f"{c1['total_trace_anomalies']} trace-anomalies across {c1['unique_trace_components_count']} components; sample components: {c1['sample_components_list']}; top trace names: {c1['top_trace_names']}" if c1 else "no trace anomalies detected")
    + "\n"
    f"- Cluster around 1616602200: "
    + (f"{c2['total_trace_anomalies']} trace-anomalies across {c2['unique_trace_components_count']} components; sample components: {c2['sample_components_list']}; top trace names: {c2['top_trace_names']}" if c2 else "no trace anomalies detected")
    + "\n\n"
    "Log preprocessing:\n"
    f"- Computed global log thresholds for {log_pairs_count} (cmdb_id, log_name) pairs (p95,p50,p5,min,max,count).\n\n"
    "Log anomalies (cluster windows ±60s):\n"
    f"- Around 1616601720: " + (f"{l1['total_log_anomalies']} log-anomalies across {l1['unique_log_components_count']} components; sample components: {l1['sample_components_list']}" if l1 else "no log anomalies detected") + "\n"
    f"- Around 1616602200: " + (f"{l2['total_log_anomalies']} log-anomalies across {l2['unique_log_components_count']} components; sample components: {l2['sample_components_list']}" if l2 else "no log anomalies detected") + "\n\n"
    "Error log check:\n"
    f"- Matching error_log records in those windows: {error_hits_count} (none found if 0).\n\n"
    "Key observations / interpretation:\n"
    "- The 1616601720 cluster shows the larger trace anomaly burst (e.g., ~158 anomalies at that minute) involving ~12 components (MG, IG, Tomcat, docker nodes). Many trace metrics (duration_mean/p95 and row_count) breached their p95/p5 thresholds.\n"
    "- The 1616602200 cluster is smaller in trace anomalies but still shows multi-component involvement.\n"
    "- Log anomalies are present but much fewer (e.g., ~11 log anomalies at 1616601720); none of the log.error_count entries are >0 in these windows (no components reported non-zero log.error_count at those exact timestamps).\n"
    "- No error_log text records matched the exact cluster timestamps ±60s, so no explicit error messages were found in error_logs.csv for these windows.\n\n"
    "Available compact outputs (already saved in kernel variables):\n"
    "- trace_global_stats  : global thresholds per (cmdb_id,trace_name)\n"
    "- trace_anom_table    : detected trace-anomaly rows (compact, up to 50 rows shown)\n"
    "- trace_agg_summary   : aggregated trace-anomaly summary per cluster timestamp\n"
    "- log_global_stats    : global thresholds per (cmdb_id,log_name)\n"
    "- log_anom_table      : detected log-anomaly rows (compact, up to 50 rows shown)\n"
    "- log_agg_summary     : aggregated log-anomaly summary per cluster timestamp\n"
    "- error_logs_hits     : matching error_logs rows (empty if none)\n\n"
    "If you want next steps, I can: (A) expand the trace/log anomaly tables for full inspection, (B) convert key anomaly lists to human-readable UTC+8 datetimes, or (C) focus on top components (e.g., MG02, IG01, Mysql01) for deeper metric-level timelines."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((261, 8),   cmdb_id                trace_name  count         p95         p50          p5   vmin        vmax
0    IG01  trace.self.duration_mean     31    0.375121    0.113987    0.052217  0.015    0.470646
1    IG01   trace.self.duration_p95     31    1.695625    1.016750    0.159375  0.015    3.053000
2    IG01      trace.self.row_count     31  606.000000  436.000000  256.000000  2.000  734.000000,    cmdb_id                       trace_name   timestamp       value         p95          p5   type
0     IG01          trace.self.duration_p95  1616601660    1.771000    1.695625    0.159375  upper
1     IG01             trace.self.row_count  1616601660  734.000000  606.000000  256.000000  upper
2     IG01      trace.to_Tomcat01.row_count  1616601660   96.000000   71.850000   33.800000  upper
3     IG01      trace.to_Tomcat02.row_count  1616601660   78.000000   76.500000   30.500000  upper
4     IG01  trace.to_Tomcat03.duration_mean  1616601660    0.431265    0.410445    0.033326  upper
..     ...                              ...         ...         ...         ...         ...    ...
45    MG02   trace.to_dockerB1.duration_p95  1616601660    2.079000    1.750230    0.052735  upper
46    MG02      trace.to_dockerB1.row_count  1616601660  141.000000  115.700000   15.000000  upper
47    MG02  trace.to_dockerB2.duration_mean  1616601660    0.341460    0.339538    0.004170  upper
48    MG02   trace.to_dockerB2.duration_p95  1616601660    2.065050    2.046037    0.008603  upper
49    MG02      trace.to_dockerB2.row_count  1616601660   50.000000   39.200000    2.000000  upper

[50 rows x 7 columns],     timestamp  total_trace_anomalies  unique_trace_components_count                          list_of_unique_components                 top_5_trace_names_by_anomaly_count
0  1616601720                    158                             12  MG02,MG01,IG01,Tomcat03,IG02,Tomcat01,dockerB1...  trace.self.duration_mean:12;trace.self.row_cou...
1  1616601660                    110                             12  MG02,MG01,dockerA2,Tomcat04,dockerB1,IG02,Tomc...  trace.self.row_count:10;trace.self.duration_p9...
2  1616602260                     45                             10  MG02,MG01,dockerA1,Tomcat01,dockerA2,Tomcat04,...  trace.to_MG02.row_count:6;trace.self.duration_...
3  1616602200                     36                             11  MG02,dockerB1,IG01,dockerA2,Tomcat02,dockerB2,...  trace.to_MG02.row_count:5;trace.from_MG02.row_...
4  1616602140                     23                              7  MG02,dockerA2,MG01,dockerA1,dockerB1,dockerB2,...  trace.to_MG02.row_count:4;trace.from_MG01.row_...
5  1616601780                     11                              7  MG02,dockerA2,Tomcat04,MG01,IG02,Tomcat03,dock...  trace.to_MG02.duration_p95:2;trace.from_IG02.r..., (20, 8),    cmdb_id         log_name  count  p95  p50   p5  vmin  vmax
0  Mysql01  log.error_count      1  0.0  0.0  0.0   0.0   0.0
1  Mysql01    log.row_count      1  1.0  1.0  1.0   1.0   1.0
2  Mysql02  log.error_count      1  0.0  0.0  0.0   0.0   0.0,      cmdb_id         log_name   timestamp  value     p95    p5   type
0   Tomcat01  log.error_count  1616601660    0.0    0.00   0.0  upper
1   Tomcat01    log.row_count  1616601660  197.0  140.85  65.4  upper
2   Tomcat02  log.error_count  1616601660    0.0    0.00   0.0  upper
3   Tomcat02    log.row_count  1616601660  172.0  156.00  66.5  upper
4   Tomcat03  log.error_count  1616601660    0.0    0.00   0.0  upper
..       ...              ...         ...    ...     ...   ...    ...
45  Tomcat01  log.error_count  1616602260    0.0    0.00   0.0  upper
46  Tomcat02  log.error_count  1616602260    0.0    0.00   0.0  upper
47  Tomcat03  log.error_count  1616602260    0.0    0.00   0.0  upper
48  Tomcat04  log.error_count  1616602260    0.0    0.00   0.0  upper
49  apache01  log.error_count  1616602260    0.0    0.00   0.0  upper

[50 rows x 7 columns],     timestamp  total_log_anomalies  unique_log_components_count                          list_of_unique_components components_with_error_count  count_components_with_error
0  1616601660                   13                            6  Tomcat04,Tomcat01,Tomcat02,Tomcat03,apache01,a...                                                        0
1  1616601720                   11                            6  Tomcat01,Tomcat02,Tomcat03,apache01,apache02,T...                                                        0
2  1616601780                    8                            6  Tomcat01,Tomcat04,Tomcat02,Tomcat03,apache01,a...                                                        0
3  1616602140                    7                            6  Tomcat04,Tomcat01,Tomcat02,Tomcat03,apache01,a...                                                        0
4  1616602200                    6                            6  Tomcat01,Tomcat02,Tomcat03,Tomcat04,apache01,a...                                                        0
5  1616602260                    6                            6  Tomcat01,Tomcat02,Tomcat03,Tomcat04,apache01,a...                                                        0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import re
import pandas as pd

# Reuse existing variables if present; otherwise load trace and trace_global_stats
# (trace should already be loaded in kernel from earlier steps)
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# 1) Ensure global trace thresholds (full-series) exist
if 'trace_global_stats' not in globals():
    trace_global_stats = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p50=lambda x: x.quantile(0.50),
        p5=lambda x: x.quantile(0.05),
        vmin='min',
        vmax='max'
    ).reset_index()
    trace_global_stats[['p95','p50','p5','vmin','vmax']] = trace_global_stats[['p95','p50','p5','vmin','vmax']].round(6)

# Cluster definitions
clusterA_center = 1616601720
clusterB_center = 1616602200
clusterA_ts = {1616601660,1616601720,1616601780}
clusterB_ts = {1616602140,1616602200,1616602260}
cluster_map = {
    'A': {'center': clusterA_center, 'ts_set': clusterA_ts},
    'B': {'center': clusterB_center, 'ts_set': clusterB_ts}
}

# 2) Recompute trace anomaly rows for the two cluster windows (join with global thresholds computed above)
stats_thr = trace_global_stats[['cmdb_id','trace_name','p95','p5']].copy()
trace_window = trace[trace['timestamp'].isin(clusterA_ts.union(clusterB_ts))].copy()
trace_win_with_stats = trace_window.merge(stats_thr, on=['cmdb_id','trace_name'], how='left')
trace_win_with_stats = trace_win_with_stats.dropna(subset=['p95','p5']).copy()
trace_win_with_stats['is_upper'] = trace_win_with_stats['value'] >= trace_win_with_stats['p95']
trace_win_with_stats['is_lower'] = trace_win_with_stats['value'] <= trace_win_with_stats['p5']
trace_win_with_stats['anomaly_type'] = pd.NA
trace_win_with_stats.loc[trace_win_with_stats['is_upper'], 'anomaly_type'] = 'upper'
trace_win_with_stats.loc[trace_win_with_stats['is_lower'] & trace_win_with_stats['anomaly_type'].isna(), 'anomaly_type'] = 'lower'
trace_anoms = trace_win_with_stats[trace_win_with_stats['anomaly_type'].notna()].copy()

# 1) Parse trace_name to extract target/source/self
def extract_components(row):
    name = row['trace_name']
    target = None
    source = None
    is_self = False
    # look for trace.to_<TARGET>. or trace.to_<TARGET>_
    m_to = re.search(r'trace\.to_([A-Za-z0-9\-]+)(?:[._]|$)', name)
    if m_to:
        target = m_to.group(1)
    m_from = re.search(r'trace\.from_([A-Za-z0-9\-]+)(?:[._]|$)', name)
    if m_from:
        source = m_from.group(1)
    if name.startswith('trace.self.') or name.startswith('trace.self_'):
        is_self = True
    return pd.Series({'target_component': target, 'source_component': source, 'is_self': is_self})

if not trace_anoms.empty:
    trace_anoms[['target_component','source_component','is_self']] = trace_anoms.apply(extract_components, axis=1)
else:
    # empty frames with expected columns
    trace_anoms[['target_component','source_component','is_self']] = pd.DataFrame(columns=['target_component','source_component','is_self'])

# Helper to aggregate per cluster
def analyze_cluster(cluster_label, center_ts, ts_set):
    # filter anomaly rows for this cluster timestamps
    df = trace_anoms[trace_anoms['timestamp'].isin(ts_set)].copy()
    df_cluster = df.copy()
    df_cluster['cluster_label'] = cluster_label
    df_cluster['timestamp_center'] = center_ts

    # A) target-aggregation: consider rows with target_component not null
    tgt_df = df_cluster[df_cluster['target_component'].notna()].copy()
    if not tgt_df.empty:
        # group by target component
        tgt_agg = tgt_df.groupby('target_component').agg(
            total_trace_anomalies_to_target = ('trace_name','size'),
            earliest_anomaly_timestamp_to_target = ('timestamp','min')
        ).reset_index()
        # compute top 3 trace_name values with counts per target
        def top3_names(group):
            top = group['trace_name'].value_counts().head(3)
            return ';'.join([f"{name}:{int(cnt)}" for name,cnt in top.items()])
        top3 = tgt_df.groupby('target_component').apply(top3_names).reset_index(name='top_trace_names_count_preview')
        tgt_agg = tgt_agg.merge(top3, on='target_component', how='left')
        tgt_agg['cluster_label'] = cluster_label
        tgt_agg['timestamp_center'] = center_ts
        tgt_agg = tgt_agg[['cluster_label','timestamp_center','target_component','total_trace_anomalies_to_target','earliest_anomaly_timestamp_to_target','top_trace_names_count_preview']]
        tgt_agg = tgt_agg.sort_values('total_trace_anomalies_to_target', ascending=False).head(20).reset_index(drop=True)
    else:
        tgt_agg = pd.DataFrame(columns=['cluster_label','timestamp_center','target_component','total_trace_anomalies_to_target','earliest_anomaly_timestamp_to_target','top_trace_names_count_preview'])

    # B) self-aggregation: trace.self.* anomalies grouped by cmdb_id
    self_df = df_cluster[df_cluster['is_self'] == True].copy()
    if not self_df.empty:
        self_agg = self_df.groupby('cmdb_id').agg(
            total_self_trace_anomalies = ('trace_name','size'),
            earliest_self_anomaly_ts = ('timestamp','min')
        ).reset_index()
        # compute self_max_breach: look at rows with duration_p95 in trace_name and compute max(value - p95)
        # Merge to get p95 per row already present; filter
        dur_p95_rows = self_df[self_df['trace_name'].str.contains('duration_p95', na=False)]
        if not dur_p95_rows.empty:
            breaches = dur_p95_rows.assign(breach = dur_p95_rows['value'] - dur_p95_rows['p95'])
            max_breach = breaches.groupby('cmdb_id')['breach'].max().reset_index(name='self_max_breach')
            self_agg = self_agg.merge(max_breach, on='cmdb_id', how='left')
        else:
            self_agg['self_max_breach'] = 0.0
        self_agg['self_max_breach'] = self_agg['self_max_breach'].fillna(0.0).astype(float)
        self_agg['cluster_label'] = cluster_label
        self_agg['timestamp_center'] = center_ts
        self_agg = self_agg[['cluster_label','timestamp_center','cmdb_id','total_self_trace_anomalies','earliest_self_anomaly_ts','self_max_breach']]
        self_agg = self_agg.sort_values('self_max_breach', ascending=False).head(20).reset_index(drop=True)
    else:
        self_agg = pd.DataFrame(columns=['cluster_label','timestamp_center','cmdb_id','total_self_trace_anomalies','earliest_self_anomaly_ts','self_max_breach'])

    # 3) Candidate downstream faulty component: choose target_component with largest total_trace_anomalies_to_target
    if not tgt_agg.empty:
        # compute tie-breaker by closeness of earliest_anomaly_timestamp to center (min abs difference)
        tgt_agg['abs_time_diff'] = (tgt_agg['earliest_anomaly_timestamp_to_target'] - center_ts).abs()
        max_count = tgt_agg['total_trace_anomalies_to_target'].max()
        candidates = tgt_agg[tgt_agg['total_trace_anomalies_to_target'] == max_count].copy()
        # pick one with minimal abs_time_diff
        candidate_row = candidates.sort_values('abs_time_diff').iloc[0]
        candidate = {
            'cluster_label': cluster_label,
            'candidate_component': candidate_row['target_component'],
            'total_trace_anomalies_to_candidate': int(candidate_row['total_trace_anomalies_to_target']),
            'earliest_anomaly_timestamp': int(candidate_row['earliest_anomaly_timestamp_to_target'])
        }
    else:
        candidate = {'cluster_label': cluster_label, 'candidate_component': None, 'total_trace_anomalies_to_candidate': 0, 'earliest_anomaly_timestamp': None}

    return tgt_agg, self_agg, candidate, df_cluster

# Analyze both clusters and store results in kernel variables
tgt_agg_A, self_agg_A, candidate_A, df_clusterA = analyze_cluster('A', clusterA_center, clusterA_ts)
tgt_agg_B, self_agg_B, candidate_B, df_clusterB = analyze_cluster('B', clusterB_center, clusterB_ts)

# Also keep a compact example of parsed anomaly rows for inspection (limit 10)
parsed_examples = trace_anoms[['cmdb_id','trace_name','timestamp','value','p95','p5','anomaly_type','target_component','source_component','is_self']].sort_values(['timestamp']).head(10).reset_index(drop=True)

# Final compact outputs (top-20 limits applied in aggregation)
tgt_agg_A.head(20), self_agg_A.head(20), candidate_A, tgt_agg_B.head(20), self_agg_B.head(20), candidate_B, parsed_examples

```
Out[6]:
```
```python
summary = (
    "Trace analysis summary (plain English):\n\n"
    "Cluster A (center 1616601720, timestamps 1616601660/1616601720/1616601780):\n"
    "- Target-aggregation: MG02 is the top referenced target (30 trace-anomalies), followed by MG01 (23). Other targets: Tomcat04, Tomcat03, dockerB1/B2, dockerA2, Tomcat01/02.\n"
    "- Top trace metrics referencing MG02 are trace.to_MG02.duration_p95 and trace.to_MG02.row_count.\n"
    "- Self-aggregation: Several components show self-metric anomalies (MG01, MG02, IG02, IG01 are prominent). MG01, MG02 and IG nodes have the largest self.duration_p95 breaches (self_max_breach ~2.9, 1.90, 1.58, 1.36 respectively).\n"
    "- Candidate downstream fault: MG02 (30 anomalies, earliest anomaly at 1616601660) — most likely downstream affected component for Cluster A.\n\n"
    "Cluster B (center 1616602200, timestamps 1616602140/1616602200/1616602260):\n"
    "- Target-aggregation: MG02 again is the top referenced target (24 trace-anomalies), then MG01 (6), dockerA1 (5), dockerB2 (4).\n"
    "- Top trace metrics referencing MG02 are trace.to_MG02.row_count and trace.to_MG02.duration_*.\n"
    "- Self-aggregation: smaller self-metric breaches; top self entries include Tomcat02, Tomcat04, docker nodes and MG02 (smaller self_max_breach values than Cluster A).\n"
    "- Candidate downstream fault: MG02 (24 anomalies, earliest anomaly at 1616602140) — most likely downstream affected component for Cluster B.\n\n"
    "Interpretation / key takeaways:\n"
    "- Both clusters show multi-component trace anomalies, but MG02 consistently appears as the top target in both windows (strong signal). MG01 is a secondary target.\n"
    "- Many anomalies are in trace.to_<component> metrics (duration_p95 and row_count), indicating increased latency/throughput towards MG02 and related services.\n"
    "- Self-metric breaches on MG and IG nodes (duration_p95) in Cluster A suggest these services experienced internal latency increases as well.\n"
    "- Recommended next steps: focus on MG02 (investigate MG02 metrics, JVM/OS, network, and downstream dependencies), then MG01 and affected Tomcat/docker services; correlate with logs and infrastructure metrics for MG02 during the exact earliest timestamps (1616601660 and 1616602140).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cluster_label  timestamp_center target_component  total_trace_anomalies_to_target  earliest_anomaly_timestamp_to_target                      top_trace_names_count_preview  abs_time_diff
0             A        1616601720             MG02                               30                            1616601660  trace.to_MG02.duration_p95:11;trace.to_MG02.ro...             60
1             A        1616601720             MG01                               23                            1616601660  trace.to_MG01.duration_mean:10;trace.to_MG01.d...             60
2             A        1616601720         Tomcat04                               10                            1616601660  trace.to_Tomcat04.row_count:4;trace.to_Tomcat0...             60
3             A        1616601720         Tomcat03                                9                            1616601660  trace.to_Tomcat03.row_count:4;trace.to_Tomcat0...             60
4             A        1616601720         dockerB1                                9                            1616601660  trace.to_dockerB1.duration_p95:4;trace.to_dock...             60
5             A        1616601720         dockerB2                                9                            1616601660  trace.to_dockerB2.duration_mean:4;trace.to_doc...             60
6             A        1616601720         dockerA2                                8                            1616601660  trace.to_dockerA2.duration_mean:4;trace.to_doc...             60
7             A        1616601720         Tomcat02                                7                            1616601660  trace.to_Tomcat02.row_count:3;trace.to_Tomcat0...             60
8             A        1616601720         Tomcat01                                7                            1616601660  trace.to_Tomcat01.row_count:3;trace.to_Tomcat0...             60
9             A        1616601720         dockerA1                                2                            1616601660  trace.to_dockerA1.duration_p95:1;trace.to_dock...             60,    cluster_label  timestamp_center   cmdb_id  total_self_trace_anomalies  earliest_self_anomaly_ts  self_max_breach
0              A        1616601720      MG01                           3                1616601660         2.855050
1              A        1616601720      MG02                           4                1616601660         1.899700
2              A        1616601720      IG02                           5                1616601660         1.579500
3              A        1616601720      IG01                           5                1616601660         1.357375
4              A        1616601720  dockerB2                           3                1616601660         0.106380
5              A        1616601720  dockerB1                           6                1616601660         0.027450
6              A        1616601720  Tomcat01                           4                1616601660         0.025745
7              A        1616601720  Tomcat04                           5                1616601660         0.023678
8              A        1616601720  Tomcat02                           4                1616601660         0.012350
9              A        1616601720  Tomcat03                           4                1616601660         0.004775
10             A        1616601720  dockerA2                           5                1616601660         0.000500
11             A        1616601720  dockerA1                           3                1616601660         0.000000, {'cluster_label': 'A', 'candidate_component': 'MG02', 'total_trace_anomalies_to_candidate': 30, 'earliest_anomaly_timestamp': 1616601660},   cluster_label  timestamp_center target_component  total_trace_anomalies_to_target  earliest_anomaly_timestamp_to_target                      top_trace_names_count_preview  abs_time_diff
0             B        1616602200             MG02                               24                            1616602140  trace.to_MG02.row_count:15;trace.to_MG02.durat...             60
1             B        1616602200             MG01                                6                            1616602140  trace.to_MG01.row_count:4;trace.to_MG01.durati...             60
2             B        1616602200         dockerA1                                5                            1616602140  trace.to_dockerA1.row_count:3;trace.to_dockerA...             60
3             B        1616602200         dockerB2                                4                            1616602140  trace.to_dockerB2.row_count:2;trace.to_dockerB...             60
4             B        1616602200         dockerA2                                3                            1616602140  trace.to_dockerA2.row_count:2;trace.to_dockerA...             60
5             B        1616602200         dockerB1                                3                            1616602140                      trace.to_dockerB1.row_count:3             60
6             B        1616602200         Tomcat02                                2                            1616602200  trace.to_Tomcat02.duration_mean:1;trace.to_Tom...              0
7             B        1616602200         Tomcat01                                1                            1616602200                      trace.to_Tomcat01.row_count:1              0,   cluster_label  timestamp_center   cmdb_id  total_self_trace_anomalies  earliest_self_anomaly_ts  self_max_breach
0             B        1616602200  Tomcat02                           1                1616602200         0.003850
1             B        1616602200  Tomcat04                           1                1616602260         0.003128
2             B        1616602200  dockerA1                           1                1616602140        -0.007000
3             B        1616602200  dockerA2                           3                1616602140        -0.007500
4             B        1616602200      MG02                           2                1616602140        -1.120100, {'cluster_label': 'B', 'candidate_component': 'MG02', 'total_trace_anomalies_to_candidate': 24, 'earliest_anomaly_timestamp': 1616602140},   cmdb_id                       trace_name   timestamp       value         p95          p5 anomaly_type target_component source_component  is_self
0    IG01          trace.self.duration_p95  1616601660    1.771000    1.695625    0.159375        upper             None             None     True
1    IG01             trace.self.row_count  1616601660  734.000000  606.000000  256.000000        upper             None             None     True
2    IG01      trace.to_Tomcat01.row_count  1616601660   96.000000   71.850000   33.800000        upper         Tomcat01             None    False
3    IG01      trace.to_Tomcat02.row_count  1616601660   78.000000   76.500000   30.500000        upper         Tomcat02             None    False
4    IG01  trace.to_Tomcat03.duration_mean  1616601660    0.431265    0.410445    0.033326        upper         Tomcat03             None    False
5    IG01      trace.to_Tomcat03.row_count  1616601660   98.000000   79.000000   25.000000        upper         Tomcat03             None    False
6    IG01      trace.to_Tomcat04.row_count  1616601660   95.000000   80.750000   30.450000        upper         Tomcat04             None    False
7    IG02          trace.self.duration_p95  1616601660    1.715000    1.547500    0.138650        upper             None             None     True
8    IG02             trace.self.row_count  1616601660  780.000000  584.000000  269.000000        upper             None             None     True
9    IG02      trace.to_Tomcat01.row_count  1616601660  101.000000   76.200000   29.900000        upper         Tomcat01             None    False)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing kernel variables where available: metric, global_stats_df, log, log_global_stats, error_logs
# Define cluster timestamp sets and components of interest
clusterA_ts = {1616601660, 1616601720, 1616601780}
clusterB_ts = {1616602140, 1616602200, 1616602260}
all_cluster_ts = clusterA_ts.union(clusterB_ts)
components = ['MG02','MG01','IG01','IG02','Mysql01','Mysql02']

# Ensure metric global stats are available (global_stats_df from earlier steps)
# global_stats_df expected to have columns: cmdb_id, kpi_name, p95, p5
stats = global_stats_df.copy()

# 1) Recompute metric-window rows for cluster timestamps and join with global thresholds
metric_window_clusters = metric[metric['timestamp'].isin(all_cluster_ts)].copy()
win = metric_window_clusters.merge(stats[['cmdb_id','kpi_name','p95','p5']], on=['cmdb_id','kpi_name'], how='left')
win = win.dropna(subset=['p95','p5']).copy()

# Mark anomaly type and compute breach (positive number)
win['is_upper'] = win['value'] >= win['p95']
win['is_lower'] = win['value'] <= win['p5']
win['anomaly_type'] = pd.NA
win.loc[win['is_upper'], 'anomaly_type'] = 'upper'
win.loc[win['is_lower'] & win['anomaly_type'].isna(), 'anomaly_type'] = 'lower'
# Keep only anomaly rows
metric_anoms = win[win['anomaly_type'].notna()].copy()
# Compute breach: for upper value - p95; for lower p5 - value
metric_anoms['breach'] = metric_anoms.apply(lambda r: (r['value'] - r['p95']) if r['anomaly_type']=='upper' else (r['p5'] - r['value']), axis=1)
# Add cluster label
def label_cluster(ts):
    if ts in clusterA_ts:
        return 'A'
    elif ts in clusterB_ts:
        return 'B'
    else:
        return None
metric_anoms['cluster_label'] = metric_anoms['timestamp'].apply(label_cluster)

# Filter to components of interest
metric_anoms_focus = metric_anoms[metric_anoms['cmdb_id'].isin(components)].copy()

# 1) Aggregation per cluster and component
if not metric_anoms_focus.empty:
    grp = metric_anoms_focus.groupby(['cluster_label','cmdb_id'])
    agg = grp.agg(
        total_anomaly_count_in_cluster_window=('timestamp','size'),
        earliest_anomaly_timestamp_in_window=('timestamp','min'),
        latest_anomaly_timestamp_in_window=('timestamp','max')
    ).reset_index()
    # top_5_kpis_by_anomaly_count
    def top5_kpis(sub):
        s = sub['kpi_name'].value_counts().head(5)
        return ';'.join([f"{k}:{int(v)}" for k,v in s.items()])
    top5 = metric_anoms_focus.groupby(['cluster_label','cmdb_id']).apply(top5_kpis).reset_index(name='top_5_kpis_by_anomaly_count')
    # max_breach among kpis
    max_breach = metric_anoms_focus.groupby(['cluster_label','cmdb_id'])['breach'].max().reset_index(name='max_breach_among_kpis')
    agg_table = agg.merge(top5, on=['cluster_label','cmdb_id'], how='left').merge(max_breach, on=['cluster_label','cmdb_id'], how='left')
    # Sort and limit to top 20
    agg_table = agg_table.sort_values(['cluster_label','total_anomaly_count_in_cluster_window','max_breach_among_kpis'], ascending=[True,False,False]).head(20).reset_index(drop=True)
else:
    agg_table = pd.DataFrame(columns=['cluster_label','cmdb_id','total_anomaly_count_in_cluster_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window','top_5_kpis_by_anomaly_count','max_breach_among_kpis'])

# 2) Up to 10 example raw metric-anomaly rows for MG02 across both clusters, sorted by breach desc
mg02_examples = metric_anoms_focus[metric_anoms_focus['cmdb_id']=='MG02'].copy()
if not mg02_examples.empty:
    mg02_examples = mg02_examples.assign(
        breach_value = mg02_examples['breach']
    ).sort_values('breach_value', ascending=False)[['cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value']].head(10).reset_index(drop=True)
else:
    mg02_examples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value'])

# 3) Search log.csv for MG02 in cluster windows and attach global log thresholds (if available)
try:
    log_stats = log_global_stats.copy()
except NameError:
    # compute if not present
    log_stats = log.groupby(['cmdb_id','log_name'])['value'].agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p50=lambda x: x.quantile(0.50),
        p5=lambda x: x.quantile(0.05),
        vmin='min',
        vmax='max'
    ).reset_index()
    log_stats[['p95','p50','p5','vmin','vmax']] = log_stats[['p95','p50','p5','vmin','vmax']].round(6)

log_mg02 = log[(log['cmdb_id']=='MG02') & (log['timestamp'].isin(all_cluster_ts))].copy()
if not log_mg02.empty:
    log_mg02 = log_mg02.merge(log_stats[['cmdb_id','log_name','p95','p5']], on=['cmdb_id','log_name'], how='left')
    log_mg02 = log_mg02[['timestamp','cmdb_id','log_name','value','p95','p5']].head(20).reset_index(drop=True)
else:
    log_mg02 = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value','p95','p5'])

# 4) Search error_logs.csv for MG02 in cluster windows
err_mg02 = error_logs[(error_logs['cmdb_id']=='MG02') & (error_logs['timestamp'].isin(all_cluster_ts))][['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# Final compact outputs saved in kernel variables (limited rows for display)
agg_table, mg02_examples, log_mg02, err_mg02

```
Out[7]:
```
```python
summary = (
    "Focused inspection summary for components [MG02, MG01, IG01, IG02, Mysql01, Mysql02] in the two cluster windows\n\n"
    "Cluster A (timestamps 1616601660, 1616601720, 1616601780):\n"
    "- Mysql02: 356 metric-anomalies (earliest 1616601660, latest 1616601780). Top KPIs include Mysql-MySQL_3306_Aborted Clients and related MySQL metrics. Max breach ~11,882.7.\n"
    "- Mysql01: 339 metric-anomalies (1616601660–1616601780). Top KPIs similar to Mysql02. Max breach ~467.5.\n"
    "- MG01: 79 metric-anomalies (1616601660–1616601780). Max breach among MG01 KPIs ~14,501,160 (large JVM/metric breach).\n"
    "- MG02: 79 metric-anomalies (1616601660–1616601780). Max breach among MG02 KPIs ~54,048.\n"
    "- IG02: 64 metric-anomalies (1616601660–1616601780). Max breach ~25.85.\n"
    "- IG01: 57 metric-anomalies (1616601660–1616601780). Max breach ~32.85.\n\n"
    "Cluster B (timestamps 1616602140, 1616602200, 1616602260):\n"
    "- Mysql01: 328 metric-anomalies (1616602140–1616602260). Max breach ~0.368.\n"
    "- Mysql02: 301 metric-anomalies (1616602140–1616602260). Max breach ~1.6.\n"
    "- MG02: 96 metric-anomalies (1616602140–1616602260). Max breach ~39,127,040 (very large breach observed).\n"
    "- MG01: 71 metric-anomalies (1616602140–1616602260). Max breach ~85,292,670 (very large breach observed).\n"
    "- IG01: 68 metric-anomalies (1616602140–1616602260). Max breach ~1.9.\n"
    "- IG02: 59 metric-anomalies (1616602140–1616602260). Max breach ~32,484,870.\n\n"
    "MG02 example metric-anomalies (top examples by breach across both clusters):\n"
    "  1) JVM-Memory_7779_JVM_Memory_HeapMemoryUsed @1616602260 — anomaly_type=lower, breach≈39,127,040 (p5 - value)\n"
    "  2) OSLinux_NETWORK_ens160_NETKBTotalPerSec @1616601720 — anomaly_type=upper, breach≈54,048 (value - p95)\n"
    "  3) OSLinux_NETWORK_ens160_NETKBTotalPerSec @1616602200 — anomaly_type=lower, breach≈7,999.6\n"
    "  4) OSLinux_NETWORK_NETWORK_TotalTcpConnNum @1616601720 — anomaly_type=upper, breach≈625\n"
    "  5) OSLinux_LOCALDISK_sdb_DSKWrite @1616601720 — anomaly_type=upper, breach≈31.55\n"
    "  (Up to 10 example rows were prepared in kernel variable 'mg02_examples'.)\n\n"
    "Logs and error logs for MG02 in these windows:\n"
    "- log.csv: no MG02 rows were found at the cluster timestamps (empty result).\n"
    "- error_logs.csv: no MG02 error_log text records at the cluster timestamps (empty result).\n\n"
    "Interpretation and recommended next steps:\n"
    "- The MySQL nodes (Mysql01/Mysql02) show very high counts of metric anomalies in both clusters — indicates broad database-related anomalies (many KPI breaches).\n"
    "- MG components (MG02 and MG01) show large magnitude metric breaches, especially in Cluster B (MG01 and MG02 have very large max_breach values). This suggests significant resource/throughput deviations (JVM memory and networking/disk metrics observed).\n"
    "- IG nodes show moderate self-metric breaches in Cluster A.\n"
    "- Network and JVM-related MG02 anomalies (NETKBTotalPerSec, TCP connections, JVM memory) are prominent in examples — investigate MG02 network, JVM memory, and disk around the earliest timestamps (1616601660 and 1616602140).\n"
    "- No direct MG02 log.error_count > 0 or textual error logs were found at exact cluster timestamps; correlate broader log ranges if needed.\n\n"
    "If you want, I can:\n"
    " - Expand the MG02 example rows into human-readable UTC+8 datetimes and full KPI names,\n"
    " - Produce time-series slices for JVM memory, network, and disk metrics for MG02/MG01 and MySQL nodes across a wider time range,\n"
    " - Or export the compact aggregation tables (agg_table, mg02_examples, log_mg02, err_mg02) to CSV/JSON for further analysis.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cluster_label  cmdb_id  total_anomaly_count_in_cluster_window  earliest_anomaly_timestamp_in_window  latest_anomaly_timestamp_in_window                        top_5_kpis_by_anomaly_count  max_breach_among_kpis
0              A  Mysql02                                    356                            1616601660                          1616601780  Mysql-MySQL_3306_Aborted Clients:3;Mysql-MySQL...           1.188267e+04
1              A  Mysql01                                    339                            1616601660                          1616601780  Mysql-MySQL_3306_Aborted Clients:3;Mysql-MySQL...           4.674583e+02
2              A     MG01                                     79                            1616601660                          1616601780  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...           1.450116e+07
3              A     MG02                                     79                            1616601660                          1616601780  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...           5.404840e+04
4              A     IG02                                     64                            1616601660                          1616601780  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:3;JVM...           2.585000e+01
5              A     IG01                                     57                            1616601660                          1616601780  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:3;JVM...           3.285000e+01
6              B  Mysql01                                    328                            1616602140                          1616602260  Mysql-MySQL_3306_Aborted Clients:3;Mysql-MySQL...           3.681450e-01
7              B  Mysql02                                    301                            1616602140                          1616602260  Mysql-MySQL_3306_Aborted Clients:3;Mysql-MySQL...           1.600000e+00
8              B     MG02                                     96                            1616602140                          1616602260  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...           3.912704e+07
9              B     MG01                                     71                            1616602140                          1616602260  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...           8.529267e+07
10             B     IG01                                     68                            1616602140                          1616602260  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:3;JVM...           1.900000e+00
11             B     IG02                                     59                            1616602140                          1616602260  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:3;JVM...           3.248487e+07,   cmdb_id                                           kpi_name   timestamp         value           p95            p5 anomaly_type  breach_value
0    MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1616602260  1.729345e+08  9.824805e+08  2.120615e+08        lower  3.912704e+07
1    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  1616601720  2.553040e+05  2.012556e+05  4.541060e+04        upper  5.404840e+04
2    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  1616602200  3.741100e+04  2.012556e+05  4.541060e+04        lower  7.999600e+03
3    MG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1616601720  2.079000e+03  1.454000e+03  2.860000e+02        upper  6.250000e+02
4    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1616601720  4.437333e+02  4.121867e+02  8.621330e+01        upper  3.154662e+01
5    MG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  1616602200  7.500000e+01  2.810476e+09  1.065000e+02        lower  3.150000e+01
6    MG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  1616602200  6.800000e+01  2.638388e+09  9.740000e+01        lower  2.940000e+01
7    MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  1616601660  7.800791e+06  7.802391e+06  7.800818e+06        lower  2.700000e+01
8    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1616601720  6.200000e+01  4.280000e+01  9.000000e+00        upper  1.920000e+01
9    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1616602200  7.613330e+01  4.121867e+02  8.621330e+01        lower  1.008000e+01, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value, p95, p5]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse existing kernel variables: metric, global_stats_df
clusterA_ts = {1616601660, 1616601720, 1616601780}
clusterB_ts = {1616602140, 1616602200, 1616602260}
all_ts = clusterA_ts.union(clusterB_ts)

# 1) Filter metric for MG02 at exact cluster minutes
mg02_window = metric[(metric['cmdb_id']=='MG02') & (metric['timestamp'].isin(all_ts))].copy()

# 2) Join with global thresholds
mg02_stats = global_stats_df[global_stats_df['cmdb_id']=='MG02'][['cmdb_id','kpi_name','p95','p5']].copy()
mg02_win = mg02_window.merge(mg02_stats, on=['cmdb_id','kpi_name'], how='left').dropna(subset=['p95','p5']).copy()

# 3) Mark anomalies and compute breach (positive magnitude)
mg02_win['is_upper'] = mg02_win['value'] >= mg02_win['p95']
mg02_win['is_lower'] = mg02_win['value'] <= mg02_win['p5']
mg02_win['anomaly_type'] = pd.NA
mg02_win.loc[mg02_win['is_upper'], 'anomaly_type'] = 'upper'
mg02_win.loc[mg02_win['is_lower'] & mg02_win['anomaly_type'].isna(), 'anomaly_type'] = 'lower'
mg02_anoms = mg02_win[mg02_win['anomaly_type'].notna()].copy()
mg02_anoms['breach_value'] = mg02_anoms.apply(
    lambda r: (r['value'] - r['p95']) if r['anomaly_type']=='upper' else (r['p5'] - r['value']),
    axis=1
).abs()

# 4) Group into fault segments where consecutive anomaly timestamps differ by <=60s
if not mg02_anoms.empty:
    mg02_anoms = mg02_anoms.sort_values('timestamp').reset_index(drop=True)
    mg02_anoms['prev_ts'] = mg02_anoms['timestamp'].shift(1)
    mg02_anoms['gap'] = mg02_anoms['timestamp'] - mg02_anoms['prev_ts']
    mg02_anoms['new_seg'] = (mg02_anoms['gap'] > 60) | (mg02_anoms['gap'].isna())
    mg02_anoms['seg_idx'] = mg02_anoms['new_seg'].cumsum()

    # assign cluster label
    def label_cluster(ts):
        if ts in clusterA_ts: return 'A'
        if ts in clusterB_ts: return 'B'
        return None
    mg02_anoms['cluster_label'] = mg02_anoms['timestamp'].apply(label_cluster)

    # segment-level aggregations
    segs = mg02_anoms.groupby('seg_idx', as_index=False).agg(
        cluster_label = ('cluster_label','first'),
        segment_start_ts = ('timestamp','min'),
        segment_end_ts = ('timestamp','max'),
        segment_length_points = ('timestamp','size')
    )

    # top 5 kpis per segment
    top5 = mg02_anoms.groupby('seg_idx').apply(lambda g: ';'.join([f"{k}:{int(v)}" for k,v in g['kpi_name'].value_counts().head(5).items()])).reset_index(name='top_5_kpis_by_anomaly_count')

    # highest breach row per segment
    idx_max = mg02_anoms.groupby('seg_idx')['breach_value'].idxmax().reset_index(name='idx')
    highest_rows = mg02_anoms.loc[idx_max['idx'], ['seg_idx','cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value']].reset_index(drop=True)

    # merge results
    segments = segs.merge(top5, on='seg_idx', how='left').merge(highest_rows, on='seg_idx', how='left')

    # prepare final segments table with requested columns
    segments_df = segments[['cluster_label','segment_start_ts','segment_end_ts','segment_length_points',
                            'top_5_kpis_by_anomaly_count',
                            'cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value']].rename(
        columns={
            'cmdb_id':'highest_breach_cmdb_id',
            'kpi_name':'highest_breach_kpi_name',
            'timestamp':'highest_breach_timestamp',
            'value':'highest_breach_value',
            'p95':'highest_breach_p95',
            'p5':'highest_breach_p5',
            'anomaly_type':'highest_breach_type',
            'breach_value':'highest_breach_value_magnitude'
        }
    ).head(20).reset_index(drop=True)
else:
    segments_df = pd.DataFrame(columns=[
        'cluster_label','segment_start_ts','segment_end_ts','segment_length_points',
        'top_5_kpis_by_anomaly_count',
        'highest_breach_cmdb_id','highest_breach_kpi_name','highest_breach_timestamp','highest_breach_value',
        'highest_breach_p95','highest_breach_p5','highest_breach_type','highest_breach_value_magnitude'
    ])

# 5) Up to 10 example MG02 anomaly rows sorted by breach_value desc
if not mg02_anoms.empty:
    mg02_examples = mg02_anoms[['cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value']].sort_values('breach_value', ascending=False).head(10).reset_index(drop=True)
else:
    mg02_examples = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value'])

# Final compact outputs (saved in kernel variables; display limited results)
segments_df, mg02_examples

```
Out[8]:
```
```python
# Plain-English summary string for MG02 root-cause confirmation
summary = (
    "MG02 focused diagnosis (compact):\n\n"
    "1) Two fault segments detected for MG02 at the analyzed cluster minutes:\n"
    "   - Segment A (cluster A): 1616601660 — 1616601780 (label A)."
    " 79 anomaly points across many KPIs in that window.\n"
    "     * Peak/most-significant anomaly (highest breach) at 1616601720: "
    "OSLinux_NETWORK_ens160_NETKBTotalPerSec (upper breach ≈ +54,048)."
    "     * Other notable anomalies in this segment: TCP connection count high (+625),"
    " local disk sdb write elevated, and various JVM/OS metrics also deviated.\n"
    "     * Interpretation: a large network throughput spike on interface ens160 around 1616601720"
    " (and related TCP connection increases) is the most likely immediate contributor to the Cluster-A impact.\n\n"
    "   - Segment B (cluster B): 1616602140 — 1616602260 (label B)."
    " 96 anomaly points across many KPIs in that window.\n"
    "     * Peak/most-significant anomaly (highest breach) at 1616602260: "
    "JVM-Memory_7779_JVM_Memory_HeapMemoryUsed (value well BELOW expected lower bound; breach ≈ 39,127,044)."
    "     * Other anomalies in this segment include reduced network throughput (NETKBTotalPerSec lower than normal) "
    "and packet/count drops at ~1616602200, and some disk metric changes.\n"
    "     * Interpretation: a large JVM heap usage change (sudden drop) at ~1616602260 suggests a JVM restart, "
    "major GC, or memory reclamation event; combined with contemporaneous network anomalies, this explains Cluster-B impact.\n\n"
    "2) Examples and evidence:\n"
    "   - Top MG02 anomaly rows (by breach) include the network throughput spike at 1616601720 and the large JVM-heap deviation at 1616602260.\n"
    "   - No MG02 entries with log.error_count > 0 or textual error_logs were found at the exact cluster minutes.\n\n"
    "3) Root-cause candidates and recommended next steps (prioritized):\n"
    "   - For Cluster A (~1616601720): investigate MG02 network (interface ens160) for spikes, NIC errors, congestion, or upstream traffic bursts. "
    "Check connection counts and network device counters around 1616601720.\n"
    "   - For Cluster B (~1616602260): investigate the MG02 JVM process for restarts or large GC events (GC logs, process restart timestamps, container restarts). "
    "The sudden large drop in HeapMemoryUsed is consistent with a restart or aggressive heap reclamation.\n"
    "   - Also check MG02 local disk (sdb) I/O and system logs (dmesg/syslog), and correlate with infrastructure events (container/host restarts, orchestration events).\n"
    "   - Correlate broader log ranges (beyond the exact minutes) for MG02 since no textual error log lines were present at the exact analyzed minutes.\n\n"
    "Conclusion: MG02 shows two distinct problem signatures: a network throughput/connection spike at ~1616601720 (Cluster A) and a JVM memory event (drop/restart/GC) at ~1616602260 (Cluster B). Both are credible root-cause candidates and should be investigated as described above."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cluster_label  segment_start_ts  segment_end_ts  segment_length_points                        top_5_kpis_by_anomaly_count highest_breach_cmdb_id                          highest_breach_kpi_name  highest_breach_timestamp  highest_breach_value  highest_breach_p95  highest_breach_p5 highest_breach_type  highest_breach_value_magnitude
0             A        1616601660      1616601780                     79  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...                   MG02  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                1616601720              255304.0            201255.6            45410.6               upper                         54048.4
1             B        1616602140      1616602260                     96  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:3;JVM...                   MG02        JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                1616602260           172934488.0         982480488.0        212061532.0               lower                      39127044.0,   cmdb_id                                           kpi_name   timestamp         value           p95            p5 anomaly_type  breach_value
0    MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1616602260  1.729345e+08  9.824805e+08  2.120615e+08        lower  3.912704e+07
1    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  1616601720  2.553040e+05  2.012556e+05  4.541060e+04        upper  5.404840e+04
2    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  1616602200  3.741100e+04  2.012556e+05  4.541060e+04        lower  7.999600e+03
3    MG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1616601720  2.079000e+03  1.454000e+03  2.860000e+02        upper  6.250000e+02
4    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1616601720  4.437333e+02  4.121867e+02  8.621330e+01        upper  3.154662e+01
5    MG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  1616602200  7.500000e+01  2.810476e+09  1.065000e+02        lower  3.150000e+01
6    MG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  1616602200  6.800000e+01  2.638388e+09  9.740000e+01        lower  2.940000e+01
7    MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  1616601660  7.800791e+06  7.802391e+06  7.800818e+06        lower  2.700000e+01
8    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1616601720  6.200000e+01  4.280000e+01  9.000000e+00        upper  1.920000e+01
9    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1616602200  7.613330e+01  4.121867e+02  8.621330e+01        lower  1.008000e+01)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime
import re
import numpy as np

# Reuse kernel variables: metric, global_stats_df, trace, trace_global_stats (should exist)
tz = pytz.timezone('Asia/Shanghai')

# Timestamps to inspect (segment start timestamps)
ts_list = [1616601660, 1616602140]
cluster_label_map = {1616601660: 'A', 1616602140: 'B'}

# 1) Convert timestamps to UTC+8 datetime strings
ts_converted = pd.DataFrame({
    'epoch_ts': ts_list,
    'cluster_label': [cluster_label_map[ts] for ts in ts_list],
    'datetime_UTC+8': [datetime.fromtimestamp(ts, pytz.utc).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z') for ts in ts_list]
})

# Helper: ensure global metric thresholds for MG02 exist
mg02_stats = global_stats_df[global_stats_df['cmdb_id']=='MG02'][['cmdb_id','kpi_name','p95','p5']].copy()

# 2) MG02 metric anomaly rows at exact timestamps
mg02_rows = metric[(metric['cmdb_id']=='MG02') & (metric['timestamp'].isin(ts_list))].copy()
mg02_rows = mg02_rows.merge(mg02_stats, on=['cmdb_id','kpi_name'], how='left').dropna(subset=['p95','p5']).copy()

# Mark anomalies and compute breach (positive magnitude)
mg02_rows['is_upper'] = mg02_rows['value'] >= mg02_rows['p95']
mg02_rows['is_lower'] = mg02_rows['value'] <= mg02_rows['p5']
mg02_rows['anomaly_type'] = np.where(mg02_rows['is_upper'],'upper', np.where(mg02_rows['is_lower'],'lower', pd.NA))
mg02_metric_anoms = mg02_rows[mg02_rows['anomaly_type'].notna()].copy()
mg02_metric_anoms['breach_value'] = mg02_metric_anoms.apply(
    lambda r: (r['value'] - r['p95']) if r['anomaly_type']=='upper' else (r['p5'] - r['value']),
    axis=1
)
# Keep requested columns and sort by breach desc (absolute positive breaches)
mg02_metric_anoms = mg02_metric_anoms[['cmdb_id','kpi_name','timestamp','value','p95','p5','anomaly_type','breach_value']].sort_values('breach_value', ascending=False).reset_index(drop=True)

# Split per timestamp (limit top 20)
mg02_metric_at_ts = {ts: mg02_metric_anoms[mg02_metric_anoms['timestamp']==ts].head(20).reset_index(drop=True) for ts in ts_list}

# 2) Trace anomalies at exact timestamps that reference MG02 as target/source or self for MG02
# Prepare trace global stats
trace_stats = trace_global_stats[['cmdb_id','trace_name','p95','p5']].copy()

# Filter trace rows at exact timestamps
trace_rows = trace[trace['timestamp'].isin(ts_list)].copy()
trace_rows = trace_rows.merge(trace_stats, on=['cmdb_id','trace_name'], how='left').dropna(subset=['p95','p5']).copy()

# Mark trace anomalies
trace_rows['is_upper'] = trace_rows['value'] >= trace_rows['p95']
trace_rows['is_lower'] = trace_rows['value'] <= trace_rows['p5']
trace_rows['anomaly_type'] = np.where(trace_rows['is_upper'],'upper', np.where(trace_rows['is_lower'],'lower', pd.NA))

# Parse trace_name to extract target/source/self
def parse_trace_components(name):
    target = None
    source = None
    is_self = False
    m_to = re.search(r'trace\.to_([A-Za-z0-9\-]+)(?:[._]|$)', name)
    if m_to:
        target = m_to.group(1)
    m_from = re.search(r'trace\.from_([A-Za-z0-9\-]+)(?:[._]|$)', name)
    if m_from:
        source = m_from.group(1)
    if name.startswith('trace.self.') or name.startswith('trace.self_'):
        is_self = True
    return pd.Series([target, source, is_self])

trace_rows[['target_component','source_component','is_self']] = trace_rows['trace_name'].apply(parse_trace_components)

# Select trace anomalies that reference MG02 (as target or source) OR self-metrics on MG02
def refs_mg02(row):
    if row['cmdb_id']=='MG02' and row['is_self']:
        return True
    if row['target_component']=='MG02' or row['source_component']=='MG02':
        return True
    return False

trace_rows['refs_MG02'] = trace_rows.apply(refs_mg02, axis=1)
trace_anoms_mg02 = trace_rows[trace_rows['anomaly_type'].notna() & trace_rows['refs_MG02']].copy()

# Keep requested columns and split per timestamp (limit top 20)
trace_anoms_mg02 = trace_anoms_mg02[['cmdb_id','trace_name','timestamp','value','p95','p5','anomaly_type']].sort_values(['timestamp','cmdb_id','trace_name']).reset_index(drop=True)
trace_anoms_at_ts = {ts: trace_anoms_mg02[trace_anoms_mg02['timestamp']==ts].head(20).reset_index(drop=True) for ts in ts_list}

# 3) Build concise 1-2 line human-readable summary per timestamp using top evidence
def infer_reason(metric_df, trace_df):
    # pick top metric kpi by breach if exists
    reason_parts = []
    if not metric_df.empty:
        top = metric_df.iloc[0]
        k = top['kpi_name'].lower()
        if 'net' in k or 'tcp' in k:
            reason_parts.append("network throughput spike on ens160; high TCP connections")
        elif 'jvm' in k or 'heap' in k:
            # check if lower breach (heap used lower) or upper
            if top['anomaly_type']=='lower':
                reason_parts.append("JVM heap drop suggesting restart/major GC")
            else:
                reason_parts.append("high JVM heap usage")
        elif 'dsk' in k or 'disk' in k:
            reason_parts.append("disk I/O spike on local disk (sda/sdb)")
        else:
            reason_parts.append(top['kpi_name'])
    # enrich with trace signals if available
    if not trace_df.empty:
        # look for trace.to_MG02 duration_p95 or row_count
        ttop = trace_df.iloc[0]['trace_name'].lower()
        if 'to_mg02' in ttop:
            if 'duration' in ttop or 'p95' in ttop:
                reason_parts.append("traces show high latency to MG02")
            if 'row_count' in ttop:
                reason_parts.append("increased throughput/requests to MG02")
        elif 'from_mg02' in ttop:
            reason_parts.append("downstream traces from MG02 affected")
        elif 'trace.self' in ttop:
            reason_parts.append("MG02 self-trace anomalies observed")
    return '; '.join(reason_parts) if reason_parts else 'anomalies observed (see evidence)'

summaries = []
for ts in ts_list:
    dt_str = datetime.fromtimestamp(ts, pytz.utc).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_df = mg02_metric_at_ts.get(ts, pd.DataFrame())
    trace_df = trace_anoms_at_ts.get(ts, pd.DataFrame())
    reason = infer_reason(metric_df, trace_df)
    summaries.append({
        'cluster_label': cluster_label_map[ts],
        'epoch_ts': int(ts),
        'datetime_UTC+8': dt_str,
        'component': 'MG02',
        'concise_reason': reason
    })

summary_df = pd.DataFrame(summaries)

# Final compact outputs (limited rows shown)
ts_converted, mg02_metric_at_ts[1616601660], trace_anoms_at_ts[1616601660], mg02_metric_at_ts[1616602140], trace_anoms_at_ts[1616602140], summary_df

```
Out[9]:
```
Summary (plain English):

Timestamps (UTC+8):
- 1616601660 → 2021-03-25 00:01:00 (UTC+8)
- 1616602140 → 2021-03-25 00:09:00 (UTC+8)

Evidence at 2021-03-25 00:01:00 (epoch 1616601660) — MG02
- Metric anomalies (top examples, sorted by breach):
  - OSLinux-OSLinux_ZABBIX_Host_Uptime — lower breach ≈ 27 sec
  - OSLinux-CPU_CPU_CPUCpuUtil — upper breach ≈ +4.32
  - OSLinux-CPU_CPU_CPULoad — upper breach ≈ +0.241
  - Various process/JVM and OS metrics flagged (some with small breaches)
- Trace anomalies (referencing MG02 at that exact minute):
  - MG02.trace.self.row_count = 633 (upper)
  - Many services (Tomcat01/02/03/04, dockerA/B) report trace.to_MG02.* and trace.from_MG02.* breaches (duration_p95, duration_mean, row_count) — mostly upper (increased latency and higher row counts)
- Interpretation (1–2 lines):
  - MG02 experienced a traffic/latency surge at ~00:01 (many upstream services show increased row_count and higher trace durations to MG02). Evidence points to an inbound throughput/latency spike (network/connection activity and high request counts toward MG02) as the immediate cause for the Cluster-A impact.

Evidence at 2021-03-25 00:09:00 (epoch 1616602140) — MG02
- Metric anomalies (top examples, sorted by breach):
  - OSLinux-CPU_CPU_CPUidleutil — small upper breach
  - OSLinux-CPU_CPU_CPUSysTime and CPUUserTime — small lower breaches
  - JVM-Threads and JVM memory meta metrics present (no large positive heap breaches at this exact minute)
  - Network TotalTcpConnNum shows lower (no large positive spike at this exact minute)
- Trace anomalies (referencing MG02 at that exact minute):
  - MG02.trace.self.row_count = 47 (lower)
  - Multiple services (Tomcat03, dockerA2, dockerB1/B2) show trace.to_MG02 or trace.from_MG02 with lower row_count and lower duration values — indicating fewer requests and reduced throughput to/from MG02
- Interpretation (1–2 lines):
  - At ~00:09 MG02 shows reduced request volume and lower trace counts toward MG02 (MG02 self row_count low). This pattern is consistent with MG02 becoming less responsive (or briefly restarted/recovered), causing downstream traffic and trace counts to drop. Combined metrics suggest a JVM/process memory event or transient outage leading to reduced throughput.

Additional notes:
- No MG02 textual error_log entries or log.error_count > 0 were found at these exact minutes.
- The two timestamps show different signatures:
  - 00:01 — inbound traffic/latency spike (high trace row_count & duration to MG02).
  - 00:09 — drop in traffic/processing at MG02 (low self row_count and lower downstream trace counts), consistent with a restart/major GC or transient unavailability.
- Recommended next steps (concise):
  - For 00:01: inspect MG02 network interface (ens160) and upstream traffic patterns, connection counts, and NIC errors around 00:01.
  - For 00:09: inspect MG02 JVM process and container/host events (GC logs, process restart times, system logs, orchestration events) around 00:09; also check surrounding minutes for correlative log entries.

The original code execution output of IPython Kernel is also provided below for reference:

(     epoch_ts cluster_label           datetime_UTC+8
0  1616601660             A  2021-03-25 00:01:00 CST
1  1616602140             B  2021-03-25 00:09:00 CST,    cmdb_id                                           kpi_name   timestamp         value           p95            p5 anomaly_type  breach_value
0     MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  1616601660  7.800791e+06  7.802391e+06  7.800818e+06        lower      27.00000
1     MG02                         OSLinux-CPU_CPU_CPUCpuUtil  1616601660  3.293850e+01  2.861965e+01  2.572874e+01        upper       4.31885
2     MG02                            OSLinux-CPU_CPU_CPULoad  1616601660  1.610000e+00  1.369000e+00  1.000000e+00        upper       0.24100
3     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  1616601660  5.893762e+07  5.893762e+07  5.893762e+07        upper       0.00000
4     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  1616601660  2.040136e+09  2.040136e+09  2.040136e+09        upper       0.00000
5     MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1616601660  1.000000e+00  1.000000e+00  1.000000e+00        upper       0.00000
6     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  1616601660  0.000000e+00  1.000000e+00  0.000000e+00        lower       0.00000
7     MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1616601660  0.000000e+00  0.000000e+00  0.000000e+00        upper       0.00000
8     MG02              OSLinux-OSLinux_MEMORY_MEMORY_UserMem  1616601660  7.164000e+03  7.369000e+03  7.164000e+03        lower       0.00000
9     MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1616601660  3.000000e+00  3.000000e+00  3.000000e+00        upper       0.00000
10    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1616601660  3.000000e-01  3.000000e-01  3.000000e-01        upper       0.00000
11    MG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  1616601660  5.800000e+01  5.800000e+01  5.800000e+01        upper       0.00000,      cmdb_id                     trace_name   timestamp       value         p95         p5 anomaly_type
0       MG02           trace.self.row_count  1616601660  633.000000  464.000000  50.000000        upper
1   Tomcat01     trace.to_MG02.duration_p95  1616601660    1.591700    1.424592   0.024858        upper
2   Tomcat01        trace.to_MG02.row_count  1616601660  107.000000   70.550000   7.350000        upper
3   Tomcat02    trace.to_MG02.duration_mean  1616601660    0.190272    0.181348   0.020631        upper
4   Tomcat02     trace.to_MG02.duration_p95  1616601660    1.045000    1.038075   0.046975        upper
5   Tomcat02        trace.to_MG02.row_count  1616601660   81.000000   79.500000   6.500000        upper
6   Tomcat03        trace.to_MG02.row_count  1616601660   88.000000   77.500000  10.000000        upper
7   Tomcat04    trace.to_MG02.duration_mean  1616601660    0.347958    0.267106   0.017756        upper
8   Tomcat04     trace.to_MG02.duration_p95  1616601660    2.058500    1.628885   0.035188        upper
9   Tomcat04        trace.to_MG02.row_count  1616601660   95.000000   81.650000  10.000000        upper
10  dockerA1   trace.from_MG02.duration_p95  1616601660    0.636600    0.627800   0.021825        upper
11  dockerA1        trace.to_MG02.row_count  1616601660   25.000000   24.650000   1.350000        upper
12  dockerA2  trace.from_MG02.duration_mean  1616601660    0.132241    0.110978   0.017516        upper
13  dockerA2   trace.from_MG02.duration_p95  1616601660    0.848650    0.611925   0.021150        upper
14  dockerB1  trace.from_MG02.duration_mean  1616601660    0.349156    0.319050   0.018544        upper
15  dockerB1   trace.from_MG02.duration_p95  1616601660    2.079000    1.750230   0.052735        upper
16  dockerB1      trace.from_MG02.row_count  1616601660  141.000000  115.700000  15.000000        upper
17  dockerB1        trace.to_MG02.row_count  1616601660  165.000000  114.300000  12.350000        upper
18  dockerB2  trace.from_MG02.duration_mean  1616601660    0.341460    0.339538   0.004170        upper
19  dockerB2   trace.from_MG02.duration_p95  1616601660    2.065050    2.046037   0.008603        upper,    cmdb_id                                           kpi_name   timestamp         value           p95            p5 anomaly_type  breach_value
0     MG02                        OSLinux-CPU_CPU_CPUidleutil  1616602140  7.445243e+01  7.433307e+01  7.204720e+01        upper      0.119358
1     MG02                         OSLinux-CPU_CPU_CPUSysTime  1616602140  2.244300e-01  5.955950e-01  2.749240e-01        lower      0.050494
2     MG02                        OSLinux-CPU_CPU_CPUUserTime  1616602140  2.513196e+01  2.562338e+01  2.516375e+01        lower      0.031799
3     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  1616602140  2.040136e+09  2.040136e+09  2.040136e+09        upper      0.000000
4     MG02           JVM-Threads_7779_JVM_ThreadCount_Threads  1616602140  7.400000e+01  7.400000e+01  7.000000e+01        upper      0.000000
5     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  1616602140  5.893762e+07  5.893762e+07  5.893762e+07        upper      0.000000
6     MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1616602140  0.000000e+00  0.000000e+00  0.000000e+00        upper      0.000000
7     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  1616602140  1.000000e+00  1.000000e+00  0.000000e+00        upper      0.000000
8     MG02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1616602140  2.860000e+02  1.454000e+03  2.860000e+02        lower      0.000000
9     MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1616602140  1.000000e+00  1.000000e+00  1.000000e+00        upper      0.000000
10    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1616602140  3.000000e-01  3.000000e-01  3.000000e-01        upper      0.000000
11    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  1616602140  3.000000e+00  3.000000e+00  3.000000e+00        upper      0.000000,     cmdb_id                  trace_name   timestamp   value         p95         p5 anomaly_type
0      MG02        trace.self.row_count  1616602140  47.000  464.000000  50.000000        lower
1  Tomcat03     trace.to_MG02.row_count  1616602140   9.000   77.500000  10.000000        lower
2  dockerA2  trace.to_MG02.duration_p95  1616602140   0.008    0.505477   0.008298        lower
3  dockerA2     trace.to_MG02.row_count  1616602140   1.000   22.650000   2.000000        lower
4  dockerB1   trace.from_MG02.row_count  1616602140  10.000  115.700000  15.000000        lower
5  dockerB1     trace.to_MG02.row_count  1616602140   6.000  114.300000  12.350000        lower
6  dockerB2   trace.from_MG02.row_count  1616602140   2.000   39.200000   2.000000        lower
7  dockerB2     trace.to_MG02.row_count  1616602140   4.000   36.150000   4.000000        lower,   cluster_label    epoch_ts           datetime_UTC+8 component                                     concise_reason
0             A  1616601660  2021-03-25 00:01:00 CST      MG02  OSLinux-OSLinux_ZABBIX_Host_Uptime; MG02 self-...
1             B  1616602140  2021-03-25 00:09:00 CST      MG02  OSLinux-CPU_CPU_CPUidleutil; MG02 self-trace a...)```
```